## 1. 필요 라이브러리 호출

In [83]:
# 환경설정
import os
import sys
import time
from tqdm import tqdm
import nest_asyncio
nest_asyncio.apply()
from dotenv import load_dotenv

load_dotenv()
# duckdb
import duckdb

# 데이터 전처리
import re
import pandas as pd
import numpy as np
import polars as pl
from datetime import datetime, timedelta
from copy import deepcopy

# 데이터 수집
import requests
from bs4 import BeautifulSoup

# 시각화
import matplotlib.pyplot as plt

# VectorDB 저장
from hashlib import md5
from langchain_community.vectorstores.utils import filter_complex_metadata # ChromaDB가 제공하지 못하는 데이터 형태를 자동으로 string처리
from datetime import datetime, timezone

## LLM 활용
# LLM 활용을 위한 dict형태 구축
from collections import defaultdict


# 1. LLM 모델 세팅 (OpenAI)
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from summart_function_openai_2 import NewsSummaryAgent, build_summary_graph  # 너가 만든 것

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.3)


# 임베딩 라이브러리
from langchain_openai import OpenAIEmbeddings # openai 
from FlagEmbedding import BGEM3FlagModel # M3
from langchain_huggingface import HuggingFaceEmbeddings

# 크로마DB
from langchain_chroma import Chroma


# ㄱRe-ranker 모델 활용
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder


# 1. 테스트셋 확인
[step]<br>

1) query 생성 : openai(gpt-3.5-turbo) 모델을 이용해서 query 생성
2) 

In [3]:
news_df = pd.read_csv('../data/korean_news_data.csv',encoding='UTF-8',index_col=[0])

In [4]:
news_df.head()

,stock,content
0,삼성전자,이재명 정부 정책 수혜주로 분류되던 고배당주 상장지수펀드(ETF)에서 자금이 빠져나...
1,삼성전자,【 앵커멘트 】이어폰을 구입할 때 주변 소음을 완벽하게 차단하는 노이즈 캔슬링이 기...
2,삼성전자,[YTN 라디오 신율의 뉴스정면승부]■ 방송 : FM 94.5 (17:00~19:0...
3,삼성전자,"삼성전자 모델이 5일 기존 식기세척기보다 에너지 소비효율 등급은 한 단계 높이고, ..."
4,삼성전자,"대표이사 직속 미래로봇추진단, AI 기반 보행·조작 기술 등 핵심 분야 인력 충원이..."


In [5]:
news_df.content[0]

'이재명 정부 정책 수혜주로 분류되던 고배당주 상장지수펀드(ETF)에서 자금이 빠져나가고 있다. 세제개편안에 담긴 배당소득 분리과세 관련 내용이 시장 기대감을 꺾은 것으로 풀이된다.5일 한국경제신문에 따르면 세제개편안이 발표된 후인 이달 1~4일 개인투자자는 국내 상장 배당 ETF 28개를 총 72억원어치 순매도했다. 올 들어 자금이 꾸준히 유입됐지만 처음으로 순유출로 전환했다.배당주 ETF 가운데 가장 규모가 큰 \'PLUS 고배당주\'는 고점 대비 9.1% 하락했다. TIGER 은행고배당플러스TOP10(-9.0%), KODEX 고배당주(-8.4%) 등 주요 배당 ETF의 낙폭도 컸다.배당주 ETF는 대표적인 정책 수혜 펀드로 분류됐다. 배당소득 분리과세가 이뤄질 것이라는 기대감에서다. 배당소득 분리과세는 투자자가 받은 배당소득을 종합과세 대상에서 제외해 분리과세하는 게 핵심이다.시장에서는 분리과세가 도입되면 기업 배당이 늘고, 투자자 세금 부담도 줄어들 것으로 기대했지만 분위기가 달라졌다.정부가 발표한 세제개편안에 따르면 배당소득 분리과세 적용 대상은 전년 대비 현금 배당이 감소하지 않은 상장법인 중 배당 성향이 40% 이상이거나, 배당 성향이 25% 이상이면서 직전 3년 대비 5% 이상 배당이 증가한 기업으로 특정됐다. 삼성전자, SK하이닉스 등 주요 종목 투자자도 혜택을 보기 어려운 조건이다.한시화 한화투자증권 연구원은 "지난 6월 대선 이후 자금이 대거 유입된 대표적인 ETF가 고배당주와 금융주"라며 "기존 개편안보다 까다로운 분리과세 방안이 발표되면서 투자심리가 악화했다"고 분석했다.(사진=연합뉴스)'

In [6]:
def querying(data):
  template = """
  <instruction>
  다음은 뉴스 기사 본문과 해당 기사에 매핑된 종목명(티커)입니다.
  당신의 임무는 뉴스 본문을 보고, 뉴스의 주요 내용을 사용자가 검색할 때 입력할 수 있는 질문을 하나 생성하세요.
  질문은 간결하고, 뉴스의 핵심 사실을 묻도록 작성하세요.

  [뉴스 본문]
  {news_content}  

  </instruction>
<example>
[뉴스 본문]
이재명 정부 정책 수혜주로 분류되던 고배당주 상장지수펀드(ETF)에서 자금이 빠져나가고 있다. 세제개편안에 담긴 배당소득 분리과세 관련 내용이 시장 기대감을 꺾은 것으로 풀이된다.5일 한국경제신문에 따르면 세제개편안이 발표된 후인 이달 1~4일 개인투자자는 국내 상장 배당 ETF 28개를 총 72억원어치 순매도했다. 올 들어 자금이 꾸준히 유입됐지만 처음으로 순유출로 전환했다.배당주 ETF 가운데 가장 규모가 큰 \'PLUS 고배당주\'는 고점 대비 9.1% 하락했다. TIGER 은행고배당플러스TOP10(-9.0%), KODEX 고배당주(-8.4%) 등 주요 배당 ETF의 낙폭도 컸다.배당주 ETF는 대표적인 정책 수혜 펀드로 분류됐다. 배당소득 분리과세가 이뤄질 것이라는 기대감에서다. 배당소득 분리과세는 투자자가 받은 배당소득을 종합과세 대상에서 제외해 분리과세하는 게 핵심이다.시장에서는 분리과세가 도입되면 기업 배당이 늘고, 투자자 세금 부담도 줄어들 것으로 기대했지만 분위기가 달라졌다.정부가 발표한 세제개편안에 따르면 배당소득 분리과세 적용 대상은 전년 대비 현금 배당이 감소하지 않은 상장법인 중 배당 성향이 40% 이상이거나, 배당 성향이 25% 이상이면서 직전 3년 대비 5% 이상 배당이 증가한 기업으로 특정됐다. 삼성전자, SK하이닉스 등 주요 종목 투자자도 혜택을 보기 어려운 조건이다.한시화 한화투자증권 연구원은 "지난 6월 대선 이후 자금이 대거 유입된 대표적인 ETF가 고배당주와 금융주"라며 "기존 개편안보다 까다로운 분리과세 방안이 발표되면서 투자심리가 악화했다"고 분석했다.(사진=연합뉴스)'

[생성한 질문]
세제개편안 발표 이후 고배당주 ETF 자금 흐름은 어떻게 변했나?
</example>

"""

  prompt = ChatPromptTemplate.from_template(template)

  # 3. 체인 구성
  chain = prompt | llm | StrOutputParser()

  # 4. 실행 예시
  query = chain.invoke({"news_content": data})
  return query

In [8]:
news_lst = []

In [9]:
for i in tqdm(range(news_df.shape[0])):
    try :
        news_dict=dict()
        news_dict['stock'] = news_df.stock[i]
        news_dict['content'] = news_df.content[i]
        news_dict['query'] = querying(news_df.content[i])
        news_lst.append(news_dict)
    except:
        news_dict=dict()
        news_dict['stock'] = news_df.stock[i]
        news_dict['content'] = news_df.content[i]
        news_dict['query'] = 'exceed'
        news_lst.append(news_dict)
        continue

100%|██████████| 5558/5558 [2:04:48<00:00,  1.35s/it]     


In [11]:
pd.DataFrame(news_lst).to_csv('pair_set.csv',encoding='utf-8')

In [2]:
pair_set = pd.read_csv('../data/news_training_df.csv',index_col=[0])

In [6]:
pair_set.stock.value_counts()

stock
Bank of Montreal                 91
Benchmark Electronics Inc        86
Empire State Realty Trust Inc    83
Inter & Co Inc                   78
PDF Solutions Inc                77
                                 ..
Caseys General Stores Inc         1
Apellis Pharmaceuticals Inc       1
Schlumberger NV                   1
Cytokinetics Inc                  1
Century Therapeutics Inc          1
Name: count, Length: 2653, dtype: int64

In [76]:
target_lst = list(pd.unique(pair_set.stock))

In [86]:
pair_set[pair_set['stock'].isin(target_lst[:10])].stock.value_counts()

stock
삼성전자         30
LG에너지솔루션     27
SK하이닉스       26
삼성바이오로직스     26
KB금융         26
두산에너빌리티      26
HD현대중공업      23
셀트리온         22
한화에어로스페이스    20
현대차          20
Name: count, dtype: int64

## 2. embedding 모델 (w/ bge-M3 모델)

In [8]:
model = BGEM3FlagModel('BAAI/bge-m3',  
                       use_fp16=True) # Setting use_fp16 to True speeds up computation with a slight performance degradation

Fetching 30 files: 100%|██████████| 30/30 [00:00<00:00, 170500.16it/s]


## 3. VectorDB 저장

In [3]:
openai_embedding = OpenAIEmbeddings(model="text-embedding-3-large")
bge_embedding = BGEM3FlagModel('BAAI/bge-m3',use_fp16=True)
persist_directory = "../VectorDB/chroma_news_db"

Fetching 30 files: 100%|██████████| 30/30 [00:00<00:00, 340078.70it/s]


In [4]:
from langchain_community.embeddings import HuggingFaceBgeEmbeddings

bge_embedding = HuggingFaceBgeEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cpu"},          # "cpu" | "cuda" | "mps"
    encode_kwargs={"normalize_embeddings": True, "batch_size": 64}
    # query_instruction / embed_instruction는 기본적으로 생략 (M3는 보통 무지시로 OK)
)

/var/folders/xq/zzsj9f116r7brgm2wm610nx00000gn/T/ipykernel_25593/1010925.py:3: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  bge_embedding = HuggingFaceBgeEmbeddings(


In [5]:
vectordb_openai = Chroma(
    persist_directory=persist_directory,
    embedding_function=openai_embedding,
    collection_name="SLM_News_openai")

In [6]:

vectordb_bge = Chroma(
    persist_directory=persist_directory,
    embedding_function=bge_embedding,
    collection_name="SLM_News_bge")

In [7]:
def labeling(data,ticker:str):
  template = """
  <instruction>
  다음은 뉴스 기사 본문과 해당 기사에 매핑된 종목명(티커)입니다.
  당신의 임무는 기사 본문이 해당 종목에 대한 기사인지 여부를 판별하는 것입니다.

  규칙:
  - 1 : 뉴스 내용이 해당 종목 기사 내용임.
     예: 종목명이 기사에 등장하고, 그 종목의 실적, 주가, 제품, 사건, 경영, 산업 동향 등과 밀접한 관련이 있음.
     예) 종목의 실적/주가/사업/이슈/계약/정책/규제/소송/리스크/전망 등.
     예) 섹터 기사라도 해당 종목이 사례/주요 구성원으로 명시적 언급되고 맥락에 기여.
  - 0 : 뉴스 내용이 해당 종목과 직접적으로 관련이 없음  
     예: 종목명이 전혀 등장하지 않거나, 비슷한 용어를 가진 단어의 내용이 등장하더라도 다른 주제가 메인인 경우.
     예) 피상적 나열(태그/키워드/꼬리말 광고)만 존재, 타사 이슈가 중심.
     
  출력 형식:
  - 숫자 1 또는 0만 출력
  </instruction>

  예시:
  ---
  [티커] 삼성전자
  [본문] 삼성전자가 2분기 실적 호조를 발표하며 주가가 3% 상승했다.
  [정답] 1
  ---
  [티커] 삼성전자
  [본문] 미국 증시가 기술주 중심으로 상승세를 보였다. 애플과 구글 주가가 상승했다.
  [정답] 0
  ---

  다음 데이터를 분류하세요.

  [티커] {ticker_name}
  [본문] {news_content}
  [정답]
  """

  prompt = ChatPromptTemplate.from_template(template)

  # 3. 체인 구성
  chain = prompt | llm | StrOutputParser()

  # 4. 실행 예시
  query = chain.invoke({"news_content": data,'ticker_name' : ticker})
  return query

In [8]:
def pick_splitter_by_length(text_len: int) -> RecursiveCharacterTextSplitter:
    """
    뉴스 본문의 길이에 따라 적절한 텍스트 분할기를 반환합니다.
    """
    if text_len <= 1200:
        # 짧은 기사 → 굳이 자르지 않고 1덩어리로 처리
        return RecursiveCharacterTextSplitter(
            chunk_size=1200,
            chunk_overlap=0,
            separators=["\n\n", "\n", " ", ""]
        )
    elif text_len <= 10_000:
        # 중간 길이 → 일반적인 1,200자 기준으로 분할
        return RecursiveCharacterTextSplitter(
            chunk_size=1200,
            chunk_overlap=150,
            separators=["\n\n", "\n", " ", ""]
        )
    elif text_len <= 50_000:
        # 긴 기사 → 덩어리를 좀 더 키움
        return RecursiveCharacterTextSplitter(
            chunk_size=1800,
            chunk_overlap=200,
            separators=["\n\n", "\n", " ", ""]
        )
    else:
        # 초장문 → 더 크게 자르되, 요약도 고려 (이건 후속 처리 필요)
        return RecursiveCharacterTextSplitter(
            chunk_size=2000,
            chunk_overlap=200,
            separators=["\n\n", "\n", " ", ""]
        )
    

In [9]:
# 2. 문서 리스트 생성 (chunk + metadata 포함)
def make_documents(df):
    docs = []

    for idx, row in tqdm(df.iterrows()):
        text = row["content"]
        splitter = pick_splitter_by_length(len(text))
        chunks = splitter.split_text(text)
        for i, chunk in enumerate(chunks):
            label = labeling(chunks,row['stock'])
            metadata = {
#               "title": row["header"],
#                "url": row["url"],
#                "Date": row["Date"],
                "ticker": row.get("stock", "None"),
                "chunk_idx": i,
                "original_idx": idx,
                'label' : label  #labeling한 결과를 넣자!! 
            }
            time.sleep(0.1)
            docs.append(Document(page_content=chunk, metadata=metadata))

    return docs


In [10]:
def make_doc_id(d: Document) -> str:
    """
    url + chunk_idx(없으면 0) + 시간
    """
    run_utc = datetime.now(timezone.utc)
    base = f"{d.metadata.get('url','')}_{d.metadata.get('chunk_idx', 0)}_{run_utc}"
    return md5(base.encode("utf-8")).hexdigest()

def chunks(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]


## Label작업 수행

In [50]:
for ticker in target_lst[:10]:
    print(f"========= ticker {ticker} 진행 중~ =============")
    #task_1_df = get_recent_articles(cusA_news_df,ticker=ticker,days = 30)
    #task_1_df['Date'] = task_1_df.Date.astype('str')
    task_1_df = pair_set[pair_set.stock==ticker]
    # document 만들기
    
    task_1_docs = make_documents(task_1_df)
    length_docs = len(task_1_docs)
    print(f'각 {ticker} 별 docs의 개수 : {length_docs}')
    BATCH = 64  # 상황에 맞게 조절

    for docs in tqdm(chunks(task_1_docs, BATCH), total=(len(task_1_docs) + BATCH - 1) // BATCH):

        ids = [make_doc_id(d) for d in docs]
        vectordb_openai.add_documents(documents=docs, ids=ids)
        time.sleep(0.1) 

========= ticker 삼성전자 진행 중~ =============


30it [00:52,  1.76s/it]


각 삼성전자 별 docs의 개수 : 59


100%|██████████| 1/1 [00:03<00:00,  3.64s/it]


========= ticker SK하이닉스 진행 중~ =============


...

100%|██████████| 1/1 [00:01<00:00,  1.99s/it]


========= ticker 셀트리온 진행 중~ =============


22it [00:31,  1.44s/it]


각 셀트리온 별 docs의 개수 : 38


100%|██████████| 1/1 [00:02<00:00,  2.78s/it]


In [18]:
for ticker in target_lst[:10]:
    print(f"========= ticker {ticker} 진행 중~ =============")
    #task_1_df = get_recent_articles(cusA_news_df,ticker=ticker,days = 30)
    #task_1_df['Date'] = task_1_df.Date.astype('str')
    task_1_df = pair_set[pair_set.stock==ticker]
    # document 만들기
    
    task_1_docs = make_documents(task_1_df)
    length_docs = len(task_1_docs)
    print(f'각 {ticker} 별 docs의 개수 : {length_docs}')
    BATCH = 64  # 상황에 맞게 조절

    for docs in tqdm(chunks(task_1_docs, BATCH), total=(len(task_1_docs) + BATCH - 1) // BATCH):
        #print(docs)
        ids = [make_doc_id(d) for d in docs]
        vectordb_bge.add_documents(documents=docs, ids=ids)
        time.sleep(0.1) 

========= ticker 삼성전자 진행 중~ =============


30it [00:49,  1.64s/it]


각 삼성전자 별 docs의 개수 : 59


  0%|          | 0/1 [00:00<?, ?it/s]/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
100%|██████████| 1/1 [01:16<00:00, 76.15s/it]


========= ticker SK하이닉스 진행 중~ =============


...

  0%|          | 0/1 [00:00<?, ?it/s]/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
100%|██████████| 1/1 [00:35<00:00, 35.86s/it]


========= ticker 셀트리온 진행 중~ =============


22it [00:30,  1.40s/it]


각 셀트리온 별 docs의 개수 : 38


  0%|          | 0/1 [00:00<?, ?it/s]/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
100%|██████████| 1/1 [00:36<00:00, 36.09s/it]


In [11]:
print("Number of documents in DB:", vectordb_openai._collection.count())

Number of documents in DB: 423


In [12]:
print("Number of documents in DB:", vectordb_bge._collection.count())

Number of documents in DB: 423


## 7. cross-encoder 모델(w/langchain 예시)

In [13]:
def _dcg_at_k(labels, k=30):
    L = np.asarray(labels)[:k]
    if L.size == 0: return 0.0
    discounts = 1.0 / np.log2(np.arange(2, L.size + 2))
    return float(np.sum(L * discounts))

def ndcg_at_k(labels, k=30):
    labels = np.asarray(labels)
    dcg  = _dcg_at_k(labels, k)
    idcg = _dcg_at_k(np.sort(labels)[::-1], k)
    return 0.0 if idcg == 0 else dcg / idcg

def precision_at_k(labels, k=30):
    L = np.asarray(labels)[:k]
    return float(L.mean()) if L.size else 0.0

def recall_at_k(labels, total_relevant, k=30):
    if not total_relevant or total_relevant <= 0: return 0.0
    return float(np.sum(np.asarray(labels)[:k])) / float(total_relevant)

def mrr_at_k(labels, k=30):
    L = np.asarray(labels)[:k]
    hit = np.where(L > 0)[0]
    return 0.0 if hit.size == 0 else 1.0 / (hit[0] + 1)

def map_at_k(labels, total_relevant, k=30):
    L = np.asarray(labels)[:k]
    if L.sum() == 0: return 0.0
    precisions, hit = [], 0
    for i, y in enumerate(L, start=1):
        if y:
            hit += 1
            precisions.append(hit / i)
    denom = max(1, min(total_relevant if total_relevant is not None else int(L.sum()), k))
    return float(np.sum(precisions) / denom)

def eval_reranker_chunk(pool_df, ranked_df, k=30, rank_col="rank", score_col="relevance_score"):
    # 1) 풀: label만 필요 (score/rank 불필요)
    total_rel_pool = int(
        pd.to_numeric(pool_df["label"], errors="coerce").fillna(0).clip(0,1).sum()
    )
    print(f'total_rel_pool : {total_rel_pool}')

    # 2) 랭크드: 순서가 필요 (rank 우선, 없으면 score로 정렬, 둘 다 없으면 현재 순서 사용)
    g = ranked_df.copy()
    if rank_col in g.columns:
        g = g.sort_values(rank_col, ascending=True)
    elif score_col in g.columns:
        g = g.sort_values(score_col, ascending=False)
        g[rank_col] = np.arange(1, len(g)+1)
    else:
        g[rank_col] = np.arange(1, len(g)+1)

    y_topk = pd.to_numeric(g["label"], errors="coerce").fillna(0).clip(0,1).astype(int).values[:k]
    print(f'y_topk : {y_topk}')
    return {
        f"Precision@{k}": precision_at_k(y_topk, k),
        f"Recall@{k}(pool)": recall_at_k(y_topk, total_rel_pool, k),
        f"MRR@{k}": mrr_at_k(y_topk, k),
        f"MAP@{k}": map_at_k(y_topk, total_rel_pool, k),
        f"nDCG@{k}": ndcg_at_k(y_topk, k),
        "PoolSize": int(len(pool_df)),
        "PoolRelevant": total_rel_pool,
        "TopK": int(min(len(g), k)),
        "TopKRelevant": int(y_topk.sum()),
    }


## Re-ranker 사용하기 전(openai)

1. 종목마다 측정

In [88]:
def pretty_print_docs(docs):
    print(
        f"\n{'-' * 100}\n".join(
            [f"Document {i + 1}:\n\n" + d.page_content for i, d in enumerate(docs)]
        )
    )

In [89]:
before_lst = []

In [90]:
for ticker in target_lst[:10]:
    retriever = vectordb_openai.as_retriever(search_kwargs={"k": 30,'filter' : {'ticker' : ticker}})
    CrossEncoder_prompt = f'''
    이 뉴스들 중에서 "{ticker}"가 핵심 주제로 다뤄진 기사만 알려줘.
    다른 회사 언급이 많거나, {ticker}가 단순히 함께 언급된 기사라면 제외해줘.
    '''
    print(CrossEncoder_prompt.strip())
    raw_docs = retriever.get_relevant_documents(CrossEncoder_prompt)
    before_lst.extend(raw_docs)
    print(list(map(lambda x: x.metadata['label'],raw_docs))[:5])
    print(list(map(lambda x: x.page_content,raw_docs))[:5])

이 뉴스들 중에서 "삼성전자"가 핵심 주제로 다뤄진 기사만 알려줘.
    다른 회사 언급이 많거나, 삼성전자가 단순히 함께 언급된 기사라면 제외해줘.
['1', '1', '1', '1', '1']
['(서울=연합뉴스) 강태우 기자 = 삼성전자가 미래로봇추진단의 잡포스팅(사내 채용공고)과 경력 채용을 동시에 실시한다. 최근 이노X 랩(InnoX Lab) 신설에 이은 내부 역량 강화 차원의 채용으로, 회사가 추진 중인 휴머노이드(인간형 로봇) 사업에도 탄력이 붙을 전망이다.삼성전자 서초사옥[연합뉴스 자료사진]5일 업계에 따르면 삼성전자 디바이스경험(DX) 부문은 이날부터 약 1주간 직원들을 대상으로 잡포스팅을 시행하는 것으로 알려졌다.모집 조직은 미래로봇추진단, 경영지원실, 경영진단팀, 생산기술연구소(생기연), 한국총괄, 디바이스 플랫폼 센터(DPC), 이사회 사무국 등이다.특히 미래로봇추진단의 잡포스팅은 전날 발표된 핵심 전략 과제 전담 조직인 이노X 랩 신설, 경력 채용과 맞물려 주목된다.미래로봇추진단은 지난해 말 삼성전자가 로봇 플랫폼 전문기업 레인보우로보틱스의 최대 주주 지위를 확보하면서 신설한 조직으로, 휴머노이드를 비롯한 삼성의 미래 로봇 기술 개발에 중추적인 역할을 수행한다.이번 잡포스팅에서는 휴머노이드향 미들웨어 개발, 인공지능(AI) 기반의 보행 및 전신 제어, 로봇 조작 기술 개발, 로봇 기구 설계 등의 인력을 미래로봇추진단에 충원하기로 했다.또 경력 채용에서는 로봇 파운데이션 모델(RFM) 개발, 로봇 조작 제어(Manipulation) 담당 인력을 뽑는 중이다.삼성전자는 올해 초에도 유관 부서 인력을 추진단에 투입하는 등 조직 역량을 내외부적으로 강화하고 있는 만큼, 휴머노이드 사업은 한층 속도가 날 전망이다.앞서 삼성전자는 자사 AI, 소프트웨어 기술에 레인보우로보틱스의 로봇 기술을 접목해 지능형 첨단 휴머노이드 개발을 가속하겠다는 계획을 밝힌 바 있다.전날부터 운영을 시작한 이노X 랩과의 유기적인 협업도 기대된다.이노X 랩은 피지컬 AI

In [91]:
rerank_df_before = pd.DataFrame(list(map(lambda x: x.metadata,before_lst)))

In [92]:
rerank_df_before

,chunk_idx,ticker,original_idx,label
0,0,삼성전자,21,1
1,1,삼성전자,22,1
2,1,삼성전자,28,1
3,0,삼성전자,4,1
4,1,삼성전자,5,1
...,...,...,...,...
295,0,셀트리온,268,1
296,0,셀트리온,281,1
297,1,셀트리온,283,1
298,1,셀트리온,285,1


In [ ]:
rerank_df_before[rerank_df_before.ticker=='삼성전자'].original_idx.value_counts() # 60%..

original_idx
36    2
11    2
28    2
10    2
5     2
2     2
14    2
30    2
22    2
34    2
15    1
20    1
7     1
21    1
25    1
13    1
24    1
9     1
4     1
18    1
Name: count, dtype: int64

In [98]:
rerank_df_before[rerank_df_before.ticker=='삼성전자'].label.astype('int').sum()

np.int64(25)

In [31]:
VecDB = vectordb_openai._collection
total_results = VecDB.get(
         include = ['documents','metadatas']   
        )

In [32]:
total_df = pd.DataFrame(total_results['metadatas'])

In [33]:
total_df

,label,ticker,chunk_idx,original_idx
0,1,삼성전자,0,0
1,1,삼성전자,0,1
2,0,삼성전자,0,2
3,0,삼성전자,1,2
4,0,삼성전자,2,2
...,...,...,...,...
418,1,셀트리온,1,285
419,1,셀트리온,0,286
420,1,셀트리온,1,286
421,1,셀트리온,0,288


In [193]:
rerank_eval = []

In [194]:
for kind in target_lst[:10]:
    total_df_target = total_df[total_df.ticker==kind]
    rerank_df_target = rerank_df_before[rerank_df_before.ticker==kind]
    rerank_eval.append(eval_reranker_chunk(total_df_target,rerank_df_target))

total_rel_pool : 34
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 0 1 1 1 1 0 1 1 0 1 0 1]
total_rel_pool : 42
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 47
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 40
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 33
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 28
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 0]
total_rel_pool : 47
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 44
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 40
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 36
y_topk : [1 1 1 0 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]


In [208]:
rerank_eval_con_before = dict(zip(target_lst[:10],rerank_eval))

In [209]:
pd.DataFrame(rerank_eval_con_before)

,삼성전자,SK하이닉스,LG에너지솔루션,삼성바이오로직스,한화에어로스페이스,현대차,KB금융,두산에너빌리티,HD현대중공업,셀트리온
Precision@30,0.833333,1.000000,1.000000,1.00,1.000000,0.933333,1.000000,1.000000,1.00,0.933333
Recall@30(pool),0.735294,0.714286,0.638298,0.75,0.909091,1.000000,0.638298,0.681818,0.75,0.777778
MRR@30,1.000000,1.000000,1.000000,1.00,1.000000,1.000000,1.000000,1.000000,1.00,1.000000
MAP@30,0.800881,1.000000,1.000000,1.00,1.000000,0.996170,1.000000,1.000000,1.00,0.827636
nDCG@30,0.991573,1.000000,1.000000,1.00,1.000000,0.999256,1.000000,1.000000,1.00,0.959072
PoolSize,59.000000,42.000000,47.000000,40.00,33.000000,31.000000,47.000000,46.000000,40.00,38.000000
PoolRelevant,34.000000,42.000000,47.000000,40.00,33.000000,28.000000,47.000000,44.000000,40.00,36.000000
TopK,30.000000,30.000000,30.000000,30.00,30.000000,30.000000,30.000000,30.000000,30.00,30.000000
TopKRelevant,25.000000,30.000000,30.000000,30.00,30.000000,28.000000,30.000000,30.000000,30.00,28.000000


2. 전체 문서에서 특정 ticker 검색

In [99]:
before_lst_total = []

In [100]:
for ticker in target_lst[:10]:
    retriever = vectordb_openai.as_retriever(search_kwargs={"k": 30})
    CrossEncoder_prompt = f'''
    이 뉴스들 중에서 "{ticker}"가 핵심 주제로 다뤄진 기사만 알려줘.
    다른 회사 언급이 많거나, {ticker}가 단순히 함께 언급된 기사라면 제외해줘.
    '''
    print(CrossEncoder_prompt.strip())
    raw_docs = retriever.get_relevant_documents(CrossEncoder_prompt)
    before_lst_total.extend(raw_docs)
    print(list(map(lambda x: x.metadata['label'],raw_docs))[:5])
    print(list(map(lambda x: x.page_content,raw_docs))[:5])

이 뉴스들 중에서 "삼성전자"가 핵심 주제로 다뤄진 기사만 알려줘.
    다른 회사 언급이 많거나, 삼성전자가 단순히 함께 언급된 기사라면 제외해줘.
['0', '1', '1', '1', '1']
['[이데일리 정두리 기자] 다음은 8월 5일자 이데일리 신문 주요 기사다.△1면-관세 못 내면 4중 처벌, 최저임금 어기면 징역 3년-LG·네이버 등 5곳 국가대표AI 만든다-신한銀, 1주택자 전세대출도 빗장-정밀·소형화 통했다 해외뻗는 K의료기기-[사설]교육세제 문제엔 눈감고 금융사 세율만 올리는 정부-[사설]조선업계 마스가 TF 가동, 민관협력 효과 극대화해야△국가대표 AI 톱5 선정-LG “프런티어 AI 활용한 생태계 구축”…네이버 “국민체험형 서비스 개발”-SKT·KT 출신 손잡은 업스테이지…스타트업 유일 골든티켓△기업 경제형벌 합리화 목소리-회계 허위 기재하면 살인죄와 같은 형량…“모기 잡는데 폭탄 쓰는 꼴”-AI교과서 교육자료로 격하…업계 “소송전 불사”-쌀 공급과잉 미리 막는다…여야 농업 4법 합의 입법△종합-소비쿠폰 노린 대형 식자재마트, 단말기 바꿔 결제…“꼼수 영업 막아야”-韓 제조업체 10곳 중 8곳 “주력제품, 이미 레드오션”-미혼남녀 60% 결혼하고 싶어도…현실 장벽은 돈-공급 늘리는 주요 산유국들 국제유가 60달러선 깨질까△의료기기 수출 특수-구부려 찍는 엑스레이, 바늘 없는 채혈기 틈새 찌르니 수출이 터졌다-“글로벌 판매량 1위 비결은 공격적 R&D”-막대한 자본 필요한 중대형 기기는 다국적기업 몫△정치-정당해산 위협에도…여론전 외엔 마땅한 대응 카드없는 국민의힘-휴가 떠난 李대통령, 韓美 정상회담·광복절 메시지 등 고심-美, 中 겨냥한 동맹 현대화 압박 가능성 커…실용외교 시험대-자본시장 신뢰 스스로 걷어찬 與△경제-“4000억이나 세수 줄어든다” 쏙 들어간 통신비 세액공제-탈세 규모 상관없이…주가 조작하면 대기업도 공시-공정위, 정책 발굴 시동…해외 경쟁당국 최신 제도 살핀다△금융-은행권, 전세·신용대출 줄줄이 중단 초강수-삼성화재,

In [101]:
rerank_df_before_total = pd.DataFrame(list(map(lambda x: x.metadata,before_lst_total)))

In [ ]:
rerank_df_before_total

,ticker,label,original_idx,chunk_idx
0,셀트리온,0,284,0
1,두산에너빌리티,1,231,0
2,SK하이닉스,1,64,0
3,LG에너지솔루션,1,99,0
4,셀트리온,1,275,0
...,...,...,...,...
295,삼성바이오로직스,1,119,0
296,LG에너지솔루션,1,88,1
297,두산에너빌리티,1,215,1
298,두산에너빌리티,1,216,0


In [ ]:
rerank_df_before_total[rerank_df_before_total.ticker == '삼성전자'].original_idx.value_counts() # 1/6 %..

original_idx
22    3
21    2
36    2
15    1
6     1
Name: count, dtype: int64

In [201]:
rerank_eval_total = []

In [202]:
for kind in target_lst[:10]:
    total_df_target = total_df[total_df.ticker==kind]
    rerank_df_target_before_total = rerank_df_before_total[rerank_df_before_total.ticker==kind]
    rerank_eval_total.append(eval_reranker_chunk(total_df_target,rerank_df_target_before_total))

total_rel_pool : 34
y_topk : [1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 42
y_topk : [1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 47
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 40
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 33
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 28
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 47
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 44
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 40
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 36
y_topk : [0 1 1 0 0 1 0 1 0 1 0 1 1 0 1 1 1 1 1 1 0 0 1 1 0 0 1 0 0 1]


In [205]:
rerank_eval_con_before_total = dict(zip(target_lst[:10],rerank_eval_total))

In [206]:
pd.DataFrame(rerank_eval_con_before_total)

,삼성전자,SK하이닉스,LG에너지솔루션,삼성바이오로직스,한화에어로스페이스,현대차,KB금융,두산에너빌리티,HD현대중공업,셀트리온
Precision@30,1.000000,1.000000,1.000000,1.00,1.000000,1.000000,1.000000,1.000000,1.00,0.566667
Recall@30(pool),0.294118,0.261905,0.595745,0.75,0.909091,0.535714,0.638298,0.681818,0.75,0.472222
MRR@30,1.000000,1.000000,1.000000,1.00,1.000000,1.000000,1.000000,1.000000,1.00,0.500000
MAP@30,0.333333,0.366667,0.933333,1.00,1.000000,0.535714,1.000000,1.000000,1.00,0.322495
nDCG@30,1.000000,1.000000,1.000000,1.00,1.000000,1.000000,1.000000,1.000000,1.00,0.771656
PoolSize,59.000000,42.000000,47.000000,40.00,33.000000,31.000000,47.000000,46.000000,40.00,38.000000
PoolRelevant,34.000000,42.000000,47.000000,40.00,33.000000,28.000000,47.000000,44.000000,40.00,36.000000
TopK,10.000000,11.000000,28.000000,30.00,30.000000,15.000000,30.000000,30.000000,30.00,30.000000
TopKRelevant,10.000000,11.000000,28.000000,30.00,30.000000,15.000000,30.000000,30.000000,30.00,17.000000


## Re-ranker 모델 사용 후 (openai)

## 1. 종목별 reranker

In [112]:
#model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")
model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")
compressor = CrossEncoderReranker(model=model, top_n=30)
#compression_retriever = ContextualCompressionRetriever(
#    base_compressor=compressor, base_retriever=retriever_total
#)

검색 유사도 측정
 - 이유 : 적정한 조건을 찾기 위함

In [113]:
scored_docs = []

In [114]:
for ticker in target_lst[:10]:
    retriever = vectordb_openai.as_retriever(search_kwargs={"k": 30,'filter' : {'ticker' : ticker}})
    CrossEncoder_prompt = f'''
    이 뉴스들 중에서 "{ticker}"가 핵심 주제로 다뤄진 기사만 알려줘.
    다른 회사 언급이 많거나, {ticker}가 단순히 함께 언급된 기사라면 제외해줘.
    '''
    print(CrossEncoder_prompt.strip())
    raw_docs = retriever.get_relevant_documents(CrossEncoder_prompt)
    pairs = [(CrossEncoder_prompt, d.page_content) for d in raw_docs]
    scores = model.score(pairs)

    # 3) 점수 붙이고 재정렬
    for d, s in zip(raw_docs, scores):
        dd = deepcopy(d)
        dd.metadata["relevance_score"] = float(s)
        scored_docs.append(dd)
    #scored_docs.sort(key=lambda x: (x.metadata[''],x.metadata["relevance_score"]), reverse=True)

이 뉴스들 중에서 "삼성전자"가 핵심 주제로 다뤄진 기사만 알려줘.
    다른 회사 언급이 많거나, 삼성전자가 단순히 함께 언급된 기사라면 제외해줘.


/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


RuntimeError: MPS backend out of memory (MPS allocated: 13.72 GB, other allocations: 3.01 GB, max allowed: 18.13 GB). Tried to allocate 1.79 GB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

In [212]:
# 4) 확인 - raw data
for i, d in enumerate(scored_docs, 1):
    if d.metadata['ticker'] == '삼성전자':
        print(f"{i:02d} | ticker : {d.metadata['ticker']}| original_idx : {d.metadata['original_idx']} | chunk_idx : {d.metadata['chunk_idx']}| score : {d.metadata['relevance_score']:.4f}")

01 | ticker : 삼성전자| original_idx : 21 | chunk_idx : 0| score : 0.0052
02 | ticker : 삼성전자| original_idx : 22 | chunk_idx : 1| score : 0.0037
03 | ticker : 삼성전자| original_idx : 28 | chunk_idx : 1| score : 0.0010
04 | ticker : 삼성전자| original_idx : 4 | chunk_idx : 0| score : 0.0018
05 | ticker : 삼성전자| original_idx : 5 | chunk_idx : 1| score : 0.0005
06 | ticker : 삼성전자| original_idx : 9 | chunk_idx : 0| score : 0.0031
07 | ticker : 삼성전자| original_idx : 24 | chunk_idx : 0| score : 0.0137
08 | ticker : 삼성전자| original_idx : 5 | chunk_idx : 0| score : 0.0016
09 | ticker : 삼성전자| original_idx : 13 | chunk_idx : 0| score : 0.0015
10 | ticker : 삼성전자| original_idx : 14 | chunk_idx : 1| score : 0.0002
11 | ticker : 삼성전자| original_idx : 30 | chunk_idx : 0| score : 0.0036
12 | ticker : 삼성전자| original_idx : 22 | chunk_idx : 0| score : 0.0001
13 | ticker : 삼성전자| original_idx : 36 | chunk_idx : 1| score : 0.0043
14 | ticker : 삼성전자| original_idx : 25 | chunk_idx : 0| score : 0.0080
15 | ticker : 삼성전자| orig

### Re-Ranker 모델 측정

In [ ]:
rerank_df = pd.DataFrame(list(map(lambda x: x.metadata,scored_docs)))

NameError: name 'scored_docs' is not defined

In [ ]:
rerank_df.ticker.value_counts()

NameError: name 'rerank_df' is not defined

In [ ]:
rerank_df.head()

,original_idx,chunk_idx,label,ticker,relevance_score
0,21,0,1,삼성전자,0.005221
1,22,1,1,삼성전자,0.003735
2,28,1,1,삼성전자,0.001007
3,4,0,1,삼성전자,0.001850
4,5,1,1,삼성전자,0.000533


In [ ]:
rerank_df= rerank_df.sort_values(by=['ticker','relevance_score'],ascending=False)

In [ ]:
rerank_df.ticker.value_counts()

ticker
현대차          30
한화에어로스페이스    30
셀트리온         30
삼성전자         30
삼성바이오로직스     30
두산에너빌리티      30
SK하이닉스       30
LG에너지솔루션     30
KB금융         30
HD현대중공업      30
Name: count, dtype: int64

In [218]:
rerank_df[rerank_df.ticker=='삼성전자'].label.value_counts()

label
1    25
0     5
Name: count, dtype: int64

In [219]:
VecDB = vectordb_openai._collection
total_results = VecDB.get(
         include = ['documents','metadatas']   
        )

In [220]:
total_df = pd.DataFrame(total_results['metadatas'])

In [221]:
total_df.shape

(423, 4)

In [222]:
total_df.ticker.value_counts()

ticker
삼성전자         59
LG에너지솔루션     47
KB금융         47
두산에너빌리티      46
SK하이닉스       42
삼성바이오로직스     40
HD현대중공업      40
셀트리온         38
한화에어로스페이스    33
현대차          31
Name: count, dtype: int64

In [223]:
total_df[total_df.ticker=='삼성전자'].label.value_counts()

label
1    34
0    25
Name: count, dtype: int64

In [224]:
rerank_eval = []

In [225]:
for kind in target_lst[:10]:
    total_df_target = total_df[total_df.ticker==kind]
    rerank_df_target = rerank_df[rerank_df.ticker==kind]
    rerank_eval.append(eval_reranker_chunk(total_df_target,rerank_df_target))

total_rel_pool : 34
y_topk : [1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 0 1 1 1 0 1 0 1 1]
total_rel_pool : 42
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 47
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 40
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 33
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 28
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 0]
total_rel_pool : 47
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 44
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 40
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 36
y_topk : [0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1]


In [226]:
rerank_eval_con = dict(zip(target_lst[:10],rerank_eval))

In [227]:
pd.DataFrame(rerank_eval_con)

,삼성전자,SK하이닉스,LG에너지솔루션,삼성바이오로직스,한화에어로스페이스,현대차,KB금융,두산에너빌리티,HD현대중공업,셀트리온
Precision@30,0.833333,1.000000,1.000000,1.00,1.000000,0.933333,1.000000,1.000000,1.00,0.933333
Recall@30(pool),0.735294,0.714286,0.638298,0.75,0.909091,1.000000,0.638298,0.681818,0.75,0.777778
MRR@30,1.000000,1.000000,1.000000,1.00,1.000000,1.000000,1.000000,1.000000,1.00,0.500000
MAP@30,0.747611,1.000000,1.000000,1.00,1.000000,0.991880,1.000000,1.000000,1.00,0.820347
nDCG@30,0.965826,1.000000,1.000000,1.00,1.000000,0.998366,1.000000,1.000000,1.00,0.905694
PoolSize,59.000000,42.000000,47.000000,40.00,33.000000,31.000000,47.000000,46.000000,40.00,38.000000
PoolRelevant,34.000000,42.000000,47.000000,40.00,33.000000,28.000000,47.000000,44.000000,40.00,36.000000
TopK,30.000000,30.000000,30.000000,30.00,30.000000,30.000000,30.000000,30.000000,30.00,30.000000
TopKRelevant,25.000000,30.000000,30.000000,30.00,30.000000,28.000000,30.000000,30.000000,30.00,28.000000


## 전체 문서에 대한 Reranking

In [241]:
import gc
gc.collect()

3178

In [244]:
scored_docs_total = []

In [245]:
for ticker in target_lst[:10]:
    retriever = vectordb_openai.as_retriever(search_kwargs={"k": 50})
    CrossEncoder_prompt = f'''
    이 뉴스들 중에서 "{ticker}"가 핵심 주제로 다뤄진 기사만 알려줘.
    다른 회사 언급이 많거나, {ticker}가 단순히 함께 언급된 기사라면 제외해줘.
    '''
    print(CrossEncoder_prompt.strip())
    raw_docs = retriever.get_relevant_documents(CrossEncoder_prompt)
    pairs = [(CrossEncoder_prompt, d.page_content) for d in raw_docs]
    scores = model.score(pairs)

    # 3) 점수 붙이고 재정렬
    for d, s in zip(raw_docs, scores):
        dd = deepcopy(d)
        dd.metadata["relevance_score"] = float(s)
        scored_docs_total.append(dd)

이 뉴스들 중에서 "삼성전자"가 핵심 주제로 다뤄진 기사만 알려줘.
    다른 회사 언급이 많거나, 삼성전자가 단순히 함께 언급된 기사라면 제외해줘.


/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


이 뉴스들 중에서 "SK하이닉스"가 핵심 주제로 다뤄진 기사만 알려줘.
    다른 회사 언급이 많거나, SK하이닉스가 단순히 함께 언급된 기사라면 제외해줘.


/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


이 뉴스들 중에서 "LG에너지솔루션"가 핵심 주제로 다뤄진 기사만 알려줘.
    다른 회사 언급이 많거나, LG에너지솔루션가 단순히 함께 언급된 기사라면 제외해줘.


...

/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


이 뉴스들 중에서 "HD현대중공업"가 핵심 주제로 다뤄진 기사만 알려줘.
    다른 회사 언급이 많거나, HD현대중공업가 단순히 함께 언급된 기사라면 제외해줘.


/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


이 뉴스들 중에서 "셀트리온"가 핵심 주제로 다뤄진 기사만 알려줘.
    다른 회사 언급이 많거나, 셀트리온가 단순히 함께 언급된 기사라면 제외해줘.


/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [21]:
rerank_df_total = pd.DataFrame(list(map(lambda x: x.metadata,scored_docs_total)))

NameError: name 'scored_docs_total' is not defined

In [250]:
rerank_df_total= rerank_df_total.sort_values(by=['ticker'],ascending=False)

In [251]:
rerank_eval_total = []

In [252]:
for kind in target_lst[:10]:
    total_df_target = total_df[total_df.ticker==kind]
    rerank_df_target_total = rerank_df_total[rerank_df_total.ticker==kind]
    rerank_eval_total.append(eval_reranker_chunk(total_df_target,rerank_df_target_total))

total_rel_pool : 34
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 0 1 1]
total_rel_pool : 42
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 47
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 40
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 33
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 28
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 47
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 44
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 40
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 36
y_topk : [1 1 1 1 0 1 1 0 1 1 1 1 1 1 1 1 1 0 0 1 1 0 1 1 1 0 1 1 1 1]


In [253]:
rerank_eval_con_total = dict(zip(target_lst[:10],rerank_eval_total))

In [254]:
pd.DataFrame(rerank_eval_con_total)

,삼성전자,SK하이닉스,LG에너지솔루션,삼성바이오로직스,한화에어로스페이스,현대차,KB금융,두산에너빌리티,HD현대중공업,셀트리온
Precision@30,0.923077,1.000000,1.000000,1.00,1.000000,1.000000,1.000000,1.000000,1.00,0.800000
Recall@30(pool),0.705882,0.523810,0.638298,0.75,0.909091,0.785714,0.638298,0.681818,0.75,0.666667
MRR@30,1.000000,1.000000,1.000000,1.00,1.000000,1.000000,1.000000,1.000000,1.00,1.000000
MAP@30,0.784945,0.733333,1.000000,1.00,1.000000,0.785714,1.000000,1.000000,1.00,0.679583
nDCG@30,0.995947,1.000000,1.000000,1.00,1.000000,1.000000,1.000000,1.000000,1.00,0.954803
PoolSize,59.000000,42.000000,47.000000,40.00,33.000000,31.000000,47.000000,46.000000,40.00,38.000000
PoolRelevant,34.000000,42.000000,47.000000,40.00,33.000000,28.000000,47.000000,44.000000,40.00,36.000000
TopK,26.000000,22.000000,30.000000,30.00,30.000000,22.000000,30.000000,30.000000,30.00,30.000000
TopKRelevant,24.000000,22.000000,30.000000,30.00,30.000000,22.000000,30.000000,30.000000,30.00,24.000000


## 사용하기 전(bge)

In [34]:
VecDB_bge = vectordb_bge._collection
total_results_bge = VecDB.get(
         include = ['documents','metadatas']   
        )

In [36]:
total_df_bge = pd.DataFrame(total_results_bge['metadatas'])

In [255]:
before_lst_bge = []

In [256]:
for ticker in target_lst[:10]:
    retriever = vectordb_bge.as_retriever(search_kwargs={"k": 30,'filter' : {'ticker' : ticker}})
    CrossEncoder_prompt = f'''
    이 뉴스들 중에서 "{ticker}"가 핵심 주제로 다뤄진 기사만 알려줘.
    다른 회사 언급이 많거나, {ticker}가 단순히 함께 언급된 기사라면 제외해줘.
    '''
    print(CrossEncoder_prompt.strip())
    raw_docs = retriever.get_relevant_documents(CrossEncoder_prompt)
    before_lst_bge.extend(raw_docs)
    print(list(map(lambda x: x.metadata['label'],raw_docs))[:5])
    print(list(map(lambda x: x.page_content,raw_docs))[:5])

이 뉴스들 중에서 "삼성전자"가 핵심 주제로 다뤄진 기사만 알려줘.
    다른 회사 언급이 많거나, 삼성전자가 단순히 함께 언급된 기사라면 제외해줘.


/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


['1', '1', '0', '1', '0']
['삼성전자 모델이 5일 기존 식기세척기보다 에너지 소비효율 등급은 한 단계 높이고, 물 사용량은 10%가량 줄인 2025년형 ‘비스포크 식기세척기 카운터탑’ 신제품을 소개하고 있다.', '하이디는 챗GPT처럼 AI 지식 검색, 영상회의 실시간 번역, 회의록 작성, 메일 요약과 초안 작성 등을 지원한다. 내부망에서 사용하기 때문에 외부로 정보가 유출되는 걸 원천적으로 차단할 수 있다. 하이디 도입으로 하루 평균 업무 생산성이 약 10% 향상됐는데, LG디스플레이는 이를 3년 내 30%까지 끌어올릴 방침이다. 외부 솔루션 대신 하이디를 자체 개발함으로써 연간 100억원에 이르는 비용 절감 효과도 거뒀다.다른 제조기업에도 AI 도입 바람이 불고 있다.조주완 LG전자 사장은 지난달 AX를 주제로 열린 임직원 소통 행사 \'AX 토크콘서트\'에 참석해 생성형 AI 데이터 시스템 \'찾다(CHATDA)\' 등을 소개했다.삼성전기는 AI를 기반으로 생산을 자동화하는 프로젝트를 진행 중이다. 이를 통해 중국 기업과 경쟁할 수 있는 생산성을 확보하겠다는 것이다.삼성전자 반도체 부문은 지난해 말 SAIT(옛 종합기술원) 산하에 있던 AI센터와 DS부문 내 혁신센터를 통합해 \'AI센터\'를 신설했다. AI를 활용해 반도체 연구개발(R&D) 역량을 높이려는 목적이다.한 국내 전자업체 CEO는 "한국보다 수십 배 많은 엔지니어를 배출하는 중국과의 경쟁에서 살아남기 위해 한국은 AI에서 답을 찾아야 한다"면서 "공장을 무인 자동화하고 AI를 활용해 R&D를 진행하는 것이 유일한 생존 방법"이라고 조언했다.[이덕주 기자]', '찾지 못했다고 보도했다. 내가 기업들, 정부, 과학자들, 변호사들, 중재자들, 피해자들과 나눈 대화에 따르면 이는 진실이 아니다"라고 썼다. 노 의원은 이 사례를 소개하면서 "이게 우리 언론의 진실"이라며 토론을 마무리했다.우원식 국회의장은 노 의원 토론을 듣고 나서 "언론의 오보를 하나하나 설명하는 게 국민들 관

/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


['1', '1', '1', '1', '1']
['삼성전자의 주가 상승과 더불어 SK하이닉스의 HBM 산업 경쟁 심화 우려로 인한 주가 조정이 함께 이뤄졌다"고 진단했다.그러면서 "내년 HBM 공급 초과분은 6%로 크지 않을 것으로 예상된다"며 "중국향 AI 칩, 소버린 AI, 네오 클라우드 등이 수요의 상방 요인으로 작용할 것"이라고 전망했다．', '건설(1.87%) 운송장비·부품(1.67%) 금속(1.54%) 등의 순으로 상승했다. 통신(-0.40%) 운송·창고(-0.35%)는 하락 마감했다.시총 상위 종목 가운데서는HLB(-0.63%)과 삼천당제약(-1.13%)이 내리는 가운데 에코프로비엠(15.28%) 에코프로(8.54%) 등 이차전지주가 급등세를 보였고 알테오젠(2.78%) 펩트론(0.34%) 파마리서치(1.62% ) 레인보우로보틱스(3.82%) 리가켐바이오(2.14%) 에이비엘바이오(5.02%) 등도 상승 마감했다.이번 주는 국내 주요 기업들의 실적 발표가 몰려있어 시장이 실적에 연동해 움직일 것으로 예상된다. 카카오, 네이버(NAVER), LG화학, SK텔레콤 등의 실적 발표가 대기 중이다.김지원 KB증권 연구원은 "최근 급등세로 고점 부담이 존재하는 만큼 실적 모멘텀이 확인돼야 현 지수대에서 추가 상승할 수 있을 것으로 보인다"고 분석했다.', '1%대 올랐다. 클래시스, 휴젤, 펩트론은 강보합권이었다. HLB는 약보합세, 삼천당제약은 1%대 약세였다.서울 외환시장에서 원/달러 환율은 이날 오후 3시30분 기준 전일 대비 3.1원 오른 1388.3원을 나타낸다.이번주 굵직한 경제 지표 발표는 없으나 국내 주요 기업들의 실적이 대거 몰려있다. 김지원 KB증권 연구원은 "에코프로, 카카오, 네이버, LG화학, SK텔레콤 등 주요 기업들이 실적 발표가 예정돼 있다"며 "최근 급등에 따른 고점 부담이 존재하는 만큼 실적 모멘텀이 확인돼야 현 지수대에서 추가 상승 가능성이 존재한다"고 짚었다.', '이차전지, 바이오주 등 성장주가 상대적 강세를 보였다”고

...

['1', '1', '1', '1', '1']
['부품 조달과 관련, 현대차 이승조 재경본부장(부사장)은 지난달 25일 2분기 실적발표 컨퍼런스콜에서 “현재 200여 개 부품에 대해 다양한 업체들로부터 견적을 받고 논의 중이며, 지속적으로 현지에서 부품 소싱 여부와 방법, 규모 등을 신중히 검토할 예정”이라고 밝혔다.이와 관련해 현대차그룹의 부울경 부품 협력업체들도 발등에 불이 떨어졌다.A사의 경우 미국 공장 설비를 배로 늘리기로 했고, 현지 공장이 없는 B사는 공장 신설을 검토중이다. 최근 현지 공장을 지은 C사는 현지 부품 공급에 어려움이 없다는 입장이다. 지역 최대 납품업체 가운데 하나인 D사 관계자는 “현대차그룹의 움직임에 따라 결정해야 할 문제이며 시장 상황을 파악 중”이라고 밝혔다.현대차그룹은 생산·부품 조달 확대와 함께 혼류생산과 로봇 확대 등을 통한 생산 효율을 높이고, 재료비·가공비 절감 노력도 함께 추진하기로 했다.', "기록했다. 연간 판매량 기준 5만대를 넘지 못할 것이라는 전망이 우세하다. 반면 하이브리드차와 전기차 판매량은 각각 20만3000여 대, 7만2000여 대로 디젤차 판매량의 10배, 3배를 기록하고 있다.디젤 차량은 중고차 시장에서도 외면받고 있다. 중고차 플랫폼 케이카가 올해 상반기 판매 데이터를 분석한 결과, 디젤차 비중은 14.9%로 지난해 같은 기간(18.4%) 대비 3.5%포인트 줄었다. 국내 수입차 시장에서도 디젤차 입지가 점점 좁아지고 있다. 올해 상반기 수입 디젤차 판매량은 1737대로 전체 판매량의 1.26%에 불과하다. 2015년 전체 수입차의 70%가량을 차지했던 디젤차는 '디젤게이트' 사건 이후로 판매량이 내리막길을 걸었다.한국수입자동차협회(KAIDA)가 집계하는 수입차 업체 26곳 중 지난 6월까지 디젤차를 판매한 업체는 아우디, BMW, 포드, 메르세데스-벤츠, 폭스바겐 등 5곳에 불과하다.[박제완 기자]", '판매량이 18만3000여 대로 디젤차 판매량(13만5000여 대)을 역전했다.올해 상반기 

/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


['1', '1', '1', '1', '1']
['많았다. 전날에는 삼성전자,한화엔진(082740), HD한국조선해양 등이 순매수 상위였으며 순매도는두산에너빌리티(034020),대한조선(439260),알테오젠(196170)순으로 많았다.미래에셋증권은 자사 고객 중에서 지난 1개월간 수익률 상위 1% 투자자들의 매매 종목을 집계해 실시간·전일·최근 5일 기준으로 모바일트레이딩시스템(MTS)상에서 공개하고 있다. 이 통계 데이터는 미래에셋증권의 의견과 무관한 단순 정보 안내이며 각각의 투자자 개인에게 맞는 투자 또는 수익 달성을 보장하지 않는다. 테마주 관련 종목은 이상 급등락 가능성이 있으므로 유의해야 한다.', '오늘(8월 5일) 오전 9시에 개장한 국내 증시에서 삼성전자(005930)가 개장 5분 만에 10.58%의 검색비율을 기록하며 많은 투자자들의 관심을 받고 있다. 삼성전자의 현재가는 71,300원으로 전 거래일 대비 2.30% 상승하며 상승세를 보이고 있다. 거래량은 1,326,241주를 기록했다.이어 두산에너빌리티(034020)가 검색비율 2위를 기록하며 0.78%의 등락률을 기록하고 있다. 검색비율 3위의 한화오션(042660)은 1.66% 상승하며 순조롭게 출발하는 모습이다. 검색비율 4위 SK하이닉스(000660)는 개장 초반부터 2.13%의 상승률로 상승 중이다. 검색비율 5위 NAVER(035420)은 0.00% 등락률로 큰 움직임을 보이지 않고 있다.6위 카카오(035720)는 등락률 -1.44%로 하락세를 보이고 있다. 7위 대한조선(439260)은 3.50%의 등락률로 주가가 소폭 상승 중이다. 8위 현대로템(064350)은 0.75%의 등락률로 상승세를 보이고 있다. 9위 알테오젠(196170)은 3.59% 상승하며 시동을 거는 모습이다. 10위 HJ중공업(097230)은 하락률 -1.08%로 주가가 다소 하락하고 있다.이 밖에도 삼성전자우 ▲4.32%, 한국전력 ▼0.05%, 에이비온 ▲2.43%, 흥구석유 ▲1.39%, 국보 ▼0

/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


['0', '1', '1', '1', '1']
['쇼크…“핀셋 정책지원 절실”-車 생산 美 현지화, 노란봉투법에 막히나-LG화학, 친환경 바이오 오일 공장 첫삽-안마해상풍력 케이블 공급 대한전선 1816억 규모 계약-오너 리스크에 실적 악화까지…분할 1년만에 위기 맞은 HS효성-삼성전자, AI·휴머노이드 전담조직 이노X랩 신설△산업-“관세폭탄 맞은 철강 살린다”…K스틸법 발의-세아베스틸, 中 특수강 봉강 반덤핑 제소-다쏘시스템, K조선 디지털 전환 혁신 이끈다-“논문 찾기 쉬워진다”…네이버, 검색결과에 통합 노출△산업-중기 수출 7분기 연속 성장…K뷰티·車가 이끌었다-동원산업, 동원F&B 자회사 편입…“해외진출 본격 시동”-CJ온스타일 더엣지 급성장…패션 독립 브랜드로 키운다△제약·바이오-새 정부 출범 후 7곳 예심청구…바이오기업 코스닥 상장 러시-“빅파마가 원하는 차별화한 신약 보유기업 찾아야”-셀트리온 유플라이마 유럽 점유율 24%…처방실적 선두 눈앞△증권-“폭락장에도 거래세는 하루 180억” 뿔난 개미들 통행세 논란 재점화-삼양컴텍·에스엔시스·에스투더블유, 조선·방산·인공지능 강세 이을까-배당소득 분리과세 너무 까다로워…배당 기대심리도 꺾였다-“변동성 장세 시작…조선·소비관광주 주목해야”△부동산-정비사업 조합 곳곳 내분에 건설사 속앓이-인왕산 아래 옥인동 4층 신축 가능해졌다△문화-낯선 생명체, 88대 스크린서 잉태하고 미술관서 낳다-한 배우의 번민 그린 무대△스포츠-해발 800m서 쾌적하게 라운드…후쿠오카로 골프여행 떠나볼까-“아직 축구선수로서 할 일 남아” 손흥민 질주, END 아닌 AND-AIG 여자오픈 4위 김아림…“아쉽지만 동기부여 생겼죠”-이현중·여준석 출격 한국 男농구…죽음의 조 뚫고 아시아컵 8강 도전△이데일리가 만났습니다-“한국 경제 복합적 위기…AI중심으로 산업·사회 대전환 나설 때”-“GDP 떨어지면 나라살림 더 악화…재정의 경기대응 역할 중요”△피플-배경훈 “모두의 AI로 나아가는 이정표 되길”-CJ대한통운, 한국 사업 인사 단행…F

In [257]:
before_lst_bge

[Document(id='1e120cef85196b2f8eb16fd8b690182c', metadata={'ticker': '삼성전자', 'original_idx': 3, 'chunk_idx': 0, 'label': '1'}, page_content='삼성전자 모델이 5일 기존 식기세척기보다 에너지 소비효율 등급은 한 단계 높이고, 물 사용량은 10%가량 줄인 2025년형 ‘비스포크 식기세척기 카운터탑’ 신제품을 소개하고 있다.'),
 Document(id='f546454692b45880c13bb0f8001e7c35', metadata={'label': '1', 'chunk_idx': 1, 'ticker': '삼성전자', 'original_idx': 22}, page_content='하이디는 챗GPT처럼 AI 지식 검색, 영상회의 실시간 번역, 회의록 작성, 메일 요약과 초안 작성 등을 지원한다. 내부망에서 사용하기 때문에 외부로 정보가 유출되는 걸 원천적으로 차단할 수 있다. 하이디 도입으로 하루 평균 업무 생산성이 약 10% 향상됐는데, LG디스플레이는 이를 3년 내 30%까지 끌어올릴 방침이다. 외부 솔루션 대신 하이디를 자체 개발함으로써 연간 100억원에 이르는 비용 절감 효과도 거뒀다.다른 제조기업에도 AI 도입 바람이 불고 있다.조주완 LG전자 사장은 지난달 AX를 주제로 열린 임직원 소통 행사 \'AX 토크콘서트\'에 참석해 생성형 AI 데이터 시스템 \'찾다(CHATDA)\' 등을 소개했다.삼성전기는 AI를 기반으로 생산을 자동화하는 프로젝트를 진행 중이다. 이를 통해 중국 기업과 경쟁할 수 있는 생산성을 확보하겠다는 것이다.삼성전자 반도체 부문은 지난해 말 SAIT(옛 종합기술원) 산하에 있던 AI센터와 DS부문 내 혁신센터를 통합해 \'AI센터\'를 신설했다. AI를 활용해 반도체 연구개발(R&D) 역량을 높이려는 목적이다.한 국내 전자업체 CEO는 "한국보다 수십 배 많은 엔지니어를 배출하는 중국과의 경쟁에서 살아남기 위해 한국은 AI에서 

In [258]:
rerank_df_before_bge = pd.DataFrame(list(map(lambda x: x.metadata,before_lst_bge)))

In [38]:
rerank_df_before_bge

NameError: name 'rerank_df_before_bge' is not defined

In [261]:
total_df = pd.DataFrame(total_results['metadatas'])

In [ ]:
rerank_eval_bge = []

In [37]:
for kind in target_lst[:10]:
    total_df_target = total_df_bge[total_df_bge.ticker==kind]
    rerank_df_target_bge = rerank_df_before_bge[rerank_df_before_bge.ticker==kind]
    rerank_eval_bge.append(eval_reranker_chunk(total_df_target,rerank_df_target_bge))

NameError: name 'rerank_df_before_bge' is not defined

In [264]:
rerank_eval_con_bge = dict(zip(target_lst[:10],rerank_eval_bge))

In [265]:
pd.DataFrame(rerank_eval_con_bge)

,삼성전자,SK하이닉스,LG에너지솔루션,삼성바이오로직스,한화에어로스페이스,현대차,KB금융,두산에너빌리티,HD현대중공업,셀트리온
Precision@30,0.633333,1.000000,1.000000,1.00,1.000000,0.900000,1.000000,0.966667,1.00,0.900000
Recall@30(pool),0.558824,0.714286,0.638298,0.75,0.909091,0.964286,0.638298,0.659091,0.75,0.750000
MRR@30,1.000000,1.000000,1.000000,1.00,1.000000,1.000000,1.000000,1.000000,1.00,0.500000
MAP@30,0.435597,1.000000,1.000000,1.00,1.000000,0.960500,1.000000,0.956528,1.00,0.750363
nDCG@30,0.884663,1.000000,1.000000,1.00,1.000000,0.999238,1.000000,0.997855,1.00,0.890603
PoolSize,59.000000,42.000000,47.000000,40.00,33.000000,31.000000,47.000000,46.000000,40.00,38.000000
PoolRelevant,34.000000,42.000000,47.000000,40.00,33.000000,28.000000,47.000000,44.000000,40.00,36.000000
TopK,30.000000,30.000000,30.000000,30.00,30.000000,30.000000,30.000000,30.000000,30.00,30.000000
TopKRelevant,19.000000,30.000000,30.000000,30.00,30.000000,27.000000,30.000000,29.000000,30.00,27.000000


## 전체 종목에 대한 검색

In [266]:
before_lst_bge_total = []

In [267]:
for ticker in target_lst[:10]:
    retriever = vectordb_bge.as_retriever(search_kwargs={"k": 30})
    CrossEncoder_prompt = f'''
    이 뉴스들 중에서 "{ticker}"가 핵심 주제로 다뤄진 기사만 알려줘.
    다른 회사 언급이 많거나, {ticker}가 단순히 함께 언급된 기사라면 제외해줘.
    '''
    print(CrossEncoder_prompt.strip())
    raw_docs = retriever.get_relevant_documents(CrossEncoder_prompt)
    before_lst_bge_total.extend(raw_docs)
    print(list(map(lambda x: x.metadata['label'],raw_docs))[:5])
    print(list(map(lambda x: x.page_content,raw_docs))[:5])

이 뉴스들 중에서 "삼성전자"가 핵심 주제로 다뤄진 기사만 알려줘.
    다른 회사 언급이 많거나, 삼성전자가 단순히 함께 언급된 기사라면 제외해줘.
['1', '1', '1', '1', '1']
['오늘(8월 5일) 오전 9시에 개장한 국내 증시에서 삼성전자(005930)가 개장 5분 만에 10.58%의 검색비율을 기록하며 많은 투자자들의 관심을 받고 있다. 삼성전자의 현재가는 71,300원으로 전 거래일 대비 2.30% 상승하며 상승세를 보이고 있다. 거래량은 1,326,241주를 기록했다.이어 두산에너빌리티(034020)가 검색비율 2위를 기록하며 0.78%의 등락률을 기록하고 있다. 검색비율 3위의 한화오션(042660)은 1.66% 상승하며 순조롭게 출발하는 모습이다. 검색비율 4위 SK하이닉스(000660)는 개장 초반부터 2.13%의 상승률로 상승 중이다. 검색비율 5위 NAVER(035420)은 0.00% 등락률로 큰 움직임을 보이지 않고 있다.6위 카카오(035720)는 등락률 -1.44%로 하락세를 보이고 있다. 7위 대한조선(439260)은 3.50%의 등락률로 주가가 소폭 상승 중이다. 8위 현대로템(064350)은 0.75%의 등락률로 상승세를 보이고 있다. 9위 알테오젠(196170)은 3.59% 상승하며 시동을 거는 모습이다. 10위 HJ중공업(097230)은 하락률 -1.08%로 주가가 다소 하락하고 있다.이 밖에도 삼성전자우 ▲4.32%, 한국전력 ▼0.05%, 에이비온 ▲2.43%, 흥구석유 ▲1.39%, 국보 ▼0.02%, 가온전선 ▼0.31%, AP헬스케어 ▼0.3%, 티에스이 ▼0.12%, 이마트 ▼0.23%, 미래산업 ▲4.44% 등이 많이 검색되고 있다.[서울신문과 MetaVX의 생성형 AI가 함께 작성한 기사입니다]', '세제 개편안은 당분간 증시 뉴스 흐름의 중심에 있을 예정”이라고 설명했다.시가총액 상위 종목 중 삼성전자(2.15%)가 7만1000원대를 회복했으며, SK하이닉스(2.13%)도 26만원대로 올라섰다

/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


['1', '1', '1', '0', '1']
['SK하이닉스(2.13%)도 26만원대로 올라섰다.아울러 LG에너지솔루션(2.26%), 삼성바이오로직스(1.24%), 현대차(1.42%), 기아(0.78%), KB금융(3.23%) 등이 오르고 있다.NAVER(-0.22%), 카카오(-2.53%) 등 인터넷 업종은 하락 중이다.업종별로 보면 증권(2.58%), 전기전자(2.21%), 화학(2.02%) 등 대다수 업종이 오르고 있으며 IT서비스(0.60%)는 하락 중이다.같은 시각 코스닥지수도 전장보다 15.97p(2.04%) 오른 800.03이다.지수는 전장보다 7.88p(1.01%) 오른 791.94로 출발해 상승폭을 확대 중이다.코스닥시장에서 외국인이 101억원 순매수하고 있으며 개인과 기관은 각각 55억원, 12억원 매도 우위를 보이고 있다.에코프로비엠(5.56%), 에코프로(4.17%) 등 이차전지주와 알테오젠(3.36%), 펩트론(1.53%), 파마리서치(2.16%) 등이 오르고 있다.HLB(-0.21%), 휴젤(-1.23%), 카카오게임즈(-0.42%) 등은 하락 중이다.', 'SK하이닉스(2.13%)도 26만원대로 올라섰다.아울러 LG에너지솔루션(2.26%), 삼성바이오로직스(1.24%), 현대차(1.42%), 기아(0.78%), KB금융(3.23%) 등이 오르고 있다.NAVER(-0.22%), 카카오(-2.53%) 등 인터넷 업종은 하락 중이다.업종별로 보면 증권(2.58%), 전기전자(2.21%), 화학(2.02%) 등 대다수 업종이 오르고 있으며 IT서비스(0.60%)는 하락 중이다.같은 시각 코스닥지수도 전장보다 15.97p(2.04%) 오른 800.03이다.지수는 전장보다 7.88p(1.01%) 오른 791.94로 출발해 상승폭을 확대 중이다.코스닥시장에서 외국인이 101억원 순매수하고 있으며 개인과 기관은 각각 55억원, 12억원 매도 우위를 보이고 있다.에코프로비엠(5.56%), 에코프로(4.17%) 등 이차전지주와 알테오젠(3.36%),

/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


['1', '1', '1', '1', '1']
['세제 개편안은 당분간 증시 뉴스 흐름의 중심에 있을 예정”이라고 설명했다.시가총액 상위 종목 중 삼성전자(2.15%)가 7만1000원대를 회복했으며, SK하이닉스(2.13%)도 26만원대로 올라섰다.아울러 LG에너지솔루션(2.26%), 삼성바이오로직스(1.24%), 현대차(1.42%), 기아(0.78%), KB금융(3.23%) 등이 오르고 있다.NAVER(-0.22%), 카카오(-2.53%) 등 인터넷 업종은 하락 중이다.업종별로 보면 증권(2.58%), 전기전자(2.21%), 화학(2.02%) 등 대다수 업종이 오르고 있으며 IT서비스(0.60%)는 하락 중이다.같은 시각 코스닥지수도 전장보다 15.97포인트(2.04%) 오른 800.03이다.지수는 전장보다 7.88포인트(1.01%) 오른 791.94로 출발해 상승폭을 확대 중이다.코스닥시장에서 외국인이 101억원 순매수하고 있으며 개인과 기관은 각각 55억원, 12억원 매도 우위를 보이고 있다.에코프로비엠(5.56%), 에코프로(4.17%) 등 이차전지주와 알테오젠(3.36%), 펩트론(1.53%), 파마리서치(2.16%) 등이 오르고 있다.HLB(-0.21%), 휴젤(-1.23%), 카카오게임즈(-0.42%) 등은 하락 중이다.', '오늘(8월 5일) 오전 9시에 개장한 국내 증시에서 삼성전자(005930)가 개장 5분 만에 10.58%의 검색비율을 기록하며 많은 투자자들의 관심을 받고 있다. 삼성전자의 현재가는 71,300원으로 전 거래일 대비 2.30% 상승하며 상승세를 보이고 있다. 거래량은 1,326,241주를 기록했다.이어 두산에너빌리티(034020)가 검색비율 2위를 기록하며 0.78%의 등락률을 기록하고 있다. 검색비율 3위의 한화오션(042660)은 1.66% 상승하며 순조롭게 출발하는 모습이다. 검색비율 4위 SK하이닉스(000660)는 개장 초반부터 2.13%의 상승률로 상승 중이다. 검색비율 5위 NAVER(035420)은 0.00% 

...

['1', '1', '0', '1', '1']
['부품 조달과 관련, 현대차 이승조 재경본부장(부사장)은 지난달 25일 2분기 실적발표 컨퍼런스콜에서 “현재 200여 개 부품에 대해 다양한 업체들로부터 견적을 받고 논의 중이며, 지속적으로 현지에서 부품 소싱 여부와 방법, 규모 등을 신중히 검토할 예정”이라고 밝혔다.이와 관련해 현대차그룹의 부울경 부품 협력업체들도 발등에 불이 떨어졌다.A사의 경우 미국 공장 설비를 배로 늘리기로 했고, 현지 공장이 없는 B사는 공장 신설을 검토중이다. 최근 현지 공장을 지은 C사는 현지 부품 공급에 어려움이 없다는 입장이다. 지역 최대 납품업체 가운데 하나인 D사 관계자는 “현대차그룹의 움직임에 따라 결정해야 할 문제이며 시장 상황을 파악 중”이라고 밝혔다.현대차그룹은 생산·부품 조달 확대와 함께 혼류생산과 로봇 확대 등을 통한 생산 효율을 높이고, 재료비·가공비 절감 노력도 함께 추진하기로 했다.', "기록했다. 연간 판매량 기준 5만대를 넘지 못할 것이라는 전망이 우세하다. 반면 하이브리드차와 전기차 판매량은 각각 20만3000여 대, 7만2000여 대로 디젤차 판매량의 10배, 3배를 기록하고 있다.디젤 차량은 중고차 시장에서도 외면받고 있다. 중고차 플랫폼 케이카가 올해 상반기 판매 데이터를 분석한 결과, 디젤차 비중은 14.9%로 지난해 같은 기간(18.4%) 대비 3.5%포인트 줄었다. 국내 수입차 시장에서도 디젤차 입지가 점점 좁아지고 있다. 올해 상반기 수입 디젤차 판매량은 1737대로 전체 판매량의 1.26%에 불과하다. 2015년 전체 수입차의 70%가량을 차지했던 디젤차는 '디젤게이트' 사건 이후로 판매량이 내리막길을 걸었다.한국수입자동차협회(KAIDA)가 집계하는 수입차 업체 26곳 중 지난 6월까지 디젤차를 판매한 업체는 아우디, BMW, 포드, 메르세데스-벤츠, 폭스바겐 등 5곳에 불과하다.[박제완 기자]", '쇼크…“핀셋 정책지원 절실”-車 생산 美 현지화, 노란봉투법에 막히나-LG화학, 친환경 바

/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


['1', '1', '1', '1', '0']
['많았다. 전날에는 삼성전자,한화엔진(082740), HD한국조선해양 등이 순매수 상위였으며 순매도는두산에너빌리티(034020),대한조선(439260),알테오젠(196170)순으로 많았다.미래에셋증권은 자사 고객 중에서 지난 1개월간 수익률 상위 1% 투자자들의 매매 종목을 집계해 실시간·전일·최근 5일 기준으로 모바일트레이딩시스템(MTS)상에서 공개하고 있다. 이 통계 데이터는 미래에셋증권의 의견과 무관한 단순 정보 안내이며 각각의 투자자 개인에게 맞는 투자 또는 수익 달성을 보장하지 않는다. 테마주 관련 종목은 이상 급등락 가능성이 있으므로 유의해야 한다.', '많았다. 전날에는 삼성전자,한화엔진(082740), HD한국조선해양 등이 순매수 상위였으며 순매도는두산에너빌리티(034020),대한조선(439260),알테오젠(196170)순으로 많았다.미래에셋증권은 자사 고객 중에서 지난 1개월간 수익률 상위 1% 투자자들의 매매 종목을 집계해 실시간·전일·최근 5일 기준으로 모바일트레이딩시스템(MTS)상에서 공개하고 있다. 이 통계 데이터는 미래에셋증권의 의견과 무관한 단순 정보 안내이며 각각의 투자자 개인에게 맞는 투자 또는 수익 달성을 보장하지 않는다. 테마주 관련 종목은 이상 급등락 가능성이 있으므로 유의해야 한다.', '많았다. 전날에는 삼성전자,한화엔진(082740), HD한국조선해양 등이 순매수 상위였으며 순매도는두산에너빌리티(034020),대한조선(439260),알테오젠(196170)순으로 많았다.미래에셋증권은 자사 고객 중에서 지난 1개월간 수익률 상위 1% 투자자들의 매매 종목을 집계해 실시간·전일·최근 5일 기준으로 모바일트레이딩시스템(MTS)상에서 공개하고 있다. 이 통계 데이터는 미래에셋증권의 의견과 무관한 단순 정보 안내이며 각각의 투자자 개인에게 맞는 투자 또는 수익 달성을 보장하지 않는다. 테마주 관련 종목은 이상 급등락 가능성이 있으므로 유의해야 한다.', '오늘(8월 5일) 오전 9시

/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


['1', '1', '0', '1', '1']
['오늘(8월 5일) 오전 9시에 개장한 국내 증시에서 삼성전자(005930)가 개장 5분 만에 10.58%의 검색비율을 기록하며 많은 투자자들의 관심을 받고 있다. 삼성전자의 현재가는 71,300원으로 전 거래일 대비 2.30% 상승하며 상승세를 보이고 있다. 거래량은 1,326,241주를 기록했다.이어 두산에너빌리티(034020)가 검색비율 2위를 기록하며 0.78%의 등락률을 기록하고 있다. 검색비율 3위의 한화오션(042660)은 1.66% 상승하며 순조롭게 출발하는 모습이다. 검색비율 4위 SK하이닉스(000660)는 개장 초반부터 2.13%의 상승률로 상승 중이다. 검색비율 5위 NAVER(035420)은 0.00% 등락률로 큰 움직임을 보이지 않고 있다.6위 카카오(035720)는 등락률 -1.44%로 하락세를 보이고 있다. 7위 대한조선(439260)은 3.50%의 등락률로 주가가 소폭 상승 중이다. 8위 현대로템(064350)은 0.75%의 등락률로 상승세를 보이고 있다. 9위 알테오젠(196170)은 3.59% 상승하며 시동을 거는 모습이다. 10위 HJ중공업(097230)은 하락률 -1.08%로 주가가 다소 하락하고 있다.이 밖에도 삼성전자우 ▲4.32%, 한국전력 ▼0.05%, 에이비온 ▲2.43%, 흥구석유 ▲1.39%, 국보 ▼0.02%, 가온전선 ▼0.31%, AP헬스케어 ▼0.3%, 티에스이 ▼0.12%, 이마트 ▼0.23%, 미래산업 ▲4.44% 등이 많이 검색되고 있다.[서울신문과 MetaVX의 생성형 AI가 함께 작성한 기사입니다]', '세제 개편안은 당분간 증시 뉴스 흐름의 중심에 있을 예정”이라고 설명했다.시가총액 상위 종목 중 삼성전자(2.15%)가 7만1000원대를 회복했으며, SK하이닉스(2.13%)도 26만원대로 올라섰다.아울러 LG에너지솔루션(2.26%), 삼성바이오로직스(1.24%), 현대차(1.42%), 기아(0.78%), KB금융(3.23%) 등이 오르고 있다.N

In [268]:
rerank_df_before_bge_total = pd.DataFrame(list(map(lambda x: x.metadata,before_lst_bge_total)))

In [270]:
rerank_eval_bge_total = []

In [271]:
for kind in target_lst[:10]:
    total_df_target = total_df[total_df.ticker==kind]
    rerank_df_target_bge_total = rerank_df_before_bge_total[rerank_df_before_bge_total.ticker==kind]
    rerank_eval_bge_total.append(eval_reranker_chunk(total_df_target,rerank_df_target_bge_total))

total_rel_pool : 34
y_topk : [1 1 1 0]
total_rel_pool : 42
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 47
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 40
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 33
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 28
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 47
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 44
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 40
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 36
y_topk : [0 1 1 1 0 1 1 1 0 1 1 1 0 1 1 1 0 1 1 0 1 1 1 1 1 0 0 1 1 1]


In [ ]:
rerank_eval_con_bge_total = dict(zip(target_lst[:10],rerank_eval_bge_total))

In [273]:
pd.DataFrame(rerank_eval_con_bge_total)

,삼성전자,SK하이닉스,LG에너지솔루션,삼성바이오로직스,한화에어로스페이스,현대차,KB금융,두산에너빌리티,HD현대중공업,셀트리온
Precision@30,0.750000,1.000000,1.000000,1.00,1.000000,1.000000,1.000000,1.000000,1.00,0.733333
Recall@30(pool),0.088235,0.476190,0.638298,0.75,0.909091,0.428571,0.553191,0.681818,0.75,0.611111
MRR@30,1.000000,1.000000,1.000000,1.00,1.000000,1.000000,1.000000,1.000000,1.00,0.500000
MAP@30,0.100000,0.666667,1.000000,1.00,1.000000,0.428571,0.866667,1.000000,1.00,0.524801
nDCG@30,1.000000,1.000000,1.000000,1.00,1.000000,1.000000,1.000000,1.000000,1.00,0.844991
PoolSize,59.000000,42.000000,47.000000,40.00,33.000000,31.000000,47.000000,46.000000,40.00,38.000000
PoolRelevant,34.000000,42.000000,47.000000,40.00,33.000000,28.000000,47.000000,44.000000,40.00,36.000000
TopK,4.000000,20.000000,30.000000,30.00,30.000000,12.000000,26.000000,30.000000,30.00,30.000000
TopKRelevant,3.000000,20.000000,30.000000,30.00,30.000000,12.000000,26.000000,30.000000,30.00,22.000000


## Reranker 모델 사용 후(bge)

1) 종목별

In [14]:
#model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")
model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")
compressor = CrossEncoderReranker(model=model, top_n=30)
#compression_retriever = ContextualCompressionRetriever(
#    base_compressor=compressor, base_retriever=retriever_total
#)

In [15]:
scored_docs_bge = []

In [290]:
import gc

In [291]:
gc.collect()

3774

In [18]:
for ticker in target_lst[:10]:
    retriever = vectordb_bge.as_retriever(search_kwargs={"k": 30,'filter' : {'ticker' : ticker}})
    CrossEncoder_prompt = f'''
    이 뉴스들 중에서 "{ticker}"가 핵심 주제로 다뤄진 기사만 알려줘.
    다른 회사 언급이 많거나, {ticker}가 단순히 함께 언급된 기사라면 제외해줘.
    '''
    print(CrossEncoder_prompt.strip())
    raw_docs = retriever.get_relevant_documents(CrossEncoder_prompt)
    pairs = [(CrossEncoder_prompt, d.page_content) for d in raw_docs]
    scores = model.score(pairs)

    # 3) 점수 붙이고 재정렬
    for d, s in zip(raw_docs, scores):
        dd = deepcopy(d)
        dd.metadata["relevance_score"] = float(s)
        scored_docs_bge.append(dd)
    #scored_docs.sort(key=lambda x: (x.metadata[''],x.metadata["relevance_score"]), reverse=True)

이 뉴스들 중에서 "삼성전자"가 핵심 주제로 다뤄진 기사만 알려줘.
    다른 회사 언급이 많거나, 삼성전자가 단순히 함께 언급된 기사라면 제외해줘.


/var/folders/xq/zzsj9f116r7brgm2wm610nx00000gn/T/ipykernel_25593/1928182108.py:8: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  raw_docs = retriever.get_relevant_documents(CrossEncoder_prompt)
/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


이 뉴스들 중에서 "SK하이닉스"가 핵심 주제로 다뤄진 기사만 알려줘.
    다른 회사 언급이 많거나, SK하이닉스가 단순히 함께 언급된 기사라면 제외해줘.


/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


이 뉴스들 중에서 "LG에너지솔루션"가 핵심 주제로 다뤄진 기사만 알려줘.
    다른 회사 언급이 많거나, LG에너지솔루션가 단순히 함께 언급된 기사라면 제외해줘.


...

/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


이 뉴스들 중에서 "HD현대중공업"가 핵심 주제로 다뤄진 기사만 알려줘.
    다른 회사 언급이 많거나, HD현대중공업가 단순히 함께 언급된 기사라면 제외해줘.


/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


이 뉴스들 중에서 "셀트리온"가 핵심 주제로 다뤄진 기사만 알려줘.
    다른 회사 언급이 많거나, 셀트리온가 단순히 함께 언급된 기사라면 제외해줘.


/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [20]:
# 4) 확인 - raw data
for i, d in enumerate(scored_docs_bge, 1):
    if d.metadata['ticker'] == '삼성전자':
        print(f"{i:02d} | ticker : {d.metadata['ticker']}| original_idx : {d.metadata['original_idx']} | chunk_idx : {d.metadata['chunk_idx']}| score : {d.metadata['relevance_score']:.4f}")

01 | ticker : 삼성전자| original_idx : 3 | chunk_idx : 0| score : 0.0011
02 | ticker : 삼성전자| original_idx : 22 | chunk_idx : 1| score : 0.0037
03 | ticker : 삼성전자| original_idx : 34 | chunk_idx : 2| score : 0.0001
04 | ticker : 삼성전자| original_idx : 5 | chunk_idx : 1| score : 0.0005
05 | ticker : 삼성전자| original_idx : 7 | chunk_idx : 6| score : 0.0010
06 | ticker : 삼성전자| original_idx : 28 | chunk_idx : 1| score : 0.0010
07 | ticker : 삼성전자| original_idx : 2 | chunk_idx : 7| score : 0.0002
08 | ticker : 삼성전자| original_idx : 2 | chunk_idx : 4| score : 0.0025
09 | ticker : 삼성전자| original_idx : 34 | chunk_idx : 1| score : 0.0065
10 | ticker : 삼성전자| original_idx : 13 | chunk_idx : 0| score : 0.0015
11 | ticker : 삼성전자| original_idx : 1 | chunk_idx : 0| score : 0.0045
12 | ticker : 삼성전자| original_idx : 36 | chunk_idx : 1| score : 0.0043
13 | ticker : 삼성전자| original_idx : 4 | chunk_idx : 0| score : 0.0018
14 | ticker : 삼성전자| original_idx : 36 | chunk_idx : 0| score : 0.0303
15 | ticker : 삼성전자| origina

In [39]:
rerank_df_bge = pd.DataFrame(list(map(lambda x: x.metadata,scored_docs_bge)))

In [40]:
rerank_df_bge= rerank_df_bge.sort_values(by=['ticker','relevance_score'],ascending=False)

In [41]:
rerank_df_bge.head()

,ticker,chunk_idx,label,original_idx,relevance_score
155,현대차,0,1,174,0.043823
153,현대차,0,1,161,0.036635
170,현대차,0,1,173,0.027360
173,현대차,0,1,176,0.015825
159,현대차,0,1,170,0.011025


In [42]:
total_df_bge

,ticker,label,chunk_idx,original_idx
0,삼성전자,1,0,0
1,삼성전자,1,0,1
2,삼성전자,0,0,2
3,삼성전자,0,1,2
4,삼성전자,0,2,2
...,...,...,...,...
418,셀트리온,1,1,285
419,셀트리온,1,0,286
420,셀트리온,1,1,286
421,셀트리온,1,0,288


In [43]:
rerank_eval_bge = []

In [45]:
for kind in target_lst[:10]:
    total_df_target = total_df_bge[total_df_bge.ticker==kind]
    rerank_df_target_bge = rerank_df_bge[rerank_df_bge.ticker==kind]
    rerank_eval_bge.append(eval_reranker_chunk(total_df_target,rerank_df_target_bge))

total_rel_pool : 34
y_topk : [1 1 0 1 1 1 1 1 1 1 1 0 1 1 1 0 0 1 0 1 1 1 1 0 0 0 0 0 1 0]
total_rel_pool : 42
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 47
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 40
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 33
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 28
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 0 1 1 1 1 0]
total_rel_pool : 47
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 44
y_topk : [1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 40
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 36
y_topk : [0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 0 1 1 1 1 1 1]


In [46]:
rerank_eval_co_bge = dict(zip(target_lst[:10],rerank_eval_bge))

In [47]:
pd.DataFrame(rerank_eval_co_bge)

,삼성전자,SK하이닉스,LG에너지솔루션,삼성바이오로직스,한화에어로스페이스,현대차,KB금융,두산에너빌리티,HD현대중공업,셀트리온
Precision@30,0.633333,1.000000,1.000000,1.00,1.000000,0.900000,1.000000,0.966667,1.00,0.90000
Recall@30(pool),0.558824,0.714286,0.638298,0.75,0.909091,0.964286,0.638298,0.659091,0.75,0.75000
MRR@30,1.000000,1.000000,1.000000,1.00,1.000000,1.000000,1.000000,1.000000,1.00,0.50000
MAP@30,0.529454,1.000000,1.000000,1.00,1.000000,0.952391,1.000000,0.936941,1.00,0.78702
nDCG@30,0.942577,1.000000,1.000000,1.00,1.000000,0.997520,1.000000,0.992367,1.00,0.90352
PoolSize,59.000000,42.000000,47.000000,40.00,33.000000,31.000000,47.000000,46.000000,40.00,38.00000
PoolRelevant,34.000000,42.000000,47.000000,40.00,33.000000,28.000000,47.000000,44.000000,40.00,36.00000
TopK,30.000000,30.000000,30.000000,30.00,30.000000,30.000000,30.000000,30.000000,30.00,30.00000
TopKRelevant,19.000000,30.000000,30.000000,30.00,30.000000,27.000000,30.000000,29.000000,30.00,27.00000


## 전체

In [62]:
scored_docs_bge_total = []

In [63]:
for ticker in target_lst[:10]:
    retriever = vectordb_bge.as_retriever(search_kwargs={"k": 50})
    CrossEncoder_prompt = f'''
    이 뉴스들 중에서 "{ticker}"가 핵심 주제로 다뤄진 기사만 알려줘.
    다른 회사 언급이 많거나, {ticker}가 단순히 함께 언급된 기사라면 제외해줘.
    '''
    print(CrossEncoder_prompt.strip())
    raw_docs = retriever.get_relevant_documents(CrossEncoder_prompt)
    pairs = [(CrossEncoder_prompt, d.page_content) for d in raw_docs]
    scores = model.score(pairs)

    # 3) 점수 붙이고 재정렬
    for d, s in zip(raw_docs, scores):
        dd = deepcopy(d)
        dd.metadata["relevance_score"] = float(s)
        scored_docs_bge_total.append(dd)
    #scored_docs.sort(key=lambda x: (x.metadata[''],x.metadata["relevance_score"]), reverse=True)

이 뉴스들 중에서 "삼성전자"가 핵심 주제로 다뤄진 기사만 알려줘.
    다른 회사 언급이 많거나, 삼성전자가 단순히 함께 언급된 기사라면 제외해줘.


/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


이 뉴스들 중에서 "SK하이닉스"가 핵심 주제로 다뤄진 기사만 알려줘.
    다른 회사 언급이 많거나, SK하이닉스가 단순히 함께 언급된 기사라면 제외해줘.


/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


이 뉴스들 중에서 "LG에너지솔루션"가 핵심 주제로 다뤄진 기사만 알려줘.
    다른 회사 언급이 많거나, LG에너지솔루션가 단순히 함께 언급된 기사라면 제외해줘.


...

/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


이 뉴스들 중에서 "HD현대중공업"가 핵심 주제로 다뤄진 기사만 알려줘.
    다른 회사 언급이 많거나, HD현대중공업가 단순히 함께 언급된 기사라면 제외해줘.


/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


이 뉴스들 중에서 "셀트리온"가 핵심 주제로 다뤄진 기사만 알려줘.
    다른 회사 언급이 많거나, 셀트리온가 단순히 함께 언급된 기사라면 제외해줘.


/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [70]:
rerank_df_bge_total = pd.DataFrame(list(map(lambda x: x.metadata,scored_docs_bge_total)))

In [71]:
rerank_df_bge_total= rerank_df_bge_total.sort_values(by=['ticker','relevance_score'],ascending=False)

In [72]:
rerank_eval_bge_total = []

In [73]:
for kind in target_lst[:10]:
    total_df_target = total_df_bge[total_df_bge.ticker==kind]
    rerank_df_target_bge_total = rerank_df_bge_total[rerank_df_bge_total.ticker==kind]
    rerank_eval_bge_total.append(eval_reranker_chunk(total_df_target,rerank_df_target_bge_total))

total_rel_pool : 34
y_topk : [1 1 1 1 1 0 0 0 0]
total_rel_pool : 42
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 47
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 40
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 33
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 28
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 47
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 44
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 40
y_topk : [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
total_rel_pool : 36
y_topk : [1 1 1 1 0 1 0 1 1 1 1 1 1 1 0 0 1 1 0 1 1 0 1 0 0 0 0 1 0 1]


In [74]:
rerank_eval_co_bge_total = dict(zip(target_lst[:10],rerank_eval_bge_total))

In [75]:
pd.DataFrame(rerank_eval_co_bge_total)

,삼성전자,SK하이닉스,LG에너지솔루션,삼성바이오로직스,한화에어로스페이스,현대차,KB금융,두산에너빌리티,HD현대중공업,셀트리온
Precision@30,0.555556,1.000000,1.000000,1.00,1.000000,1.000000,1.000000,1.000000,1.00,0.633333
Recall@30(pool),0.147059,0.714286,0.638298,0.75,0.909091,0.678571,0.638298,0.681818,0.75,0.527778
MRR@30,1.000000,1.000000,1.000000,1.00,1.000000,1.000000,1.000000,1.000000,1.00,1.000000
MAP@30,0.166667,1.000000,1.000000,1.00,1.000000,0.678571,1.000000,1.000000,1.00,0.519521
nDCG@30,1.000000,1.000000,1.000000,1.00,1.000000,1.000000,1.000000,1.000000,1.00,0.945909
PoolSize,59.000000,42.000000,47.000000,40.00,33.000000,31.000000,47.000000,46.000000,40.00,38.000000
PoolRelevant,34.000000,42.000000,47.000000,40.00,33.000000,28.000000,47.000000,44.000000,40.00,36.000000
TopK,9.000000,30.000000,30.000000,30.00,30.000000,19.000000,30.000000,30.000000,30.00,30.000000
TopKRelevant,5.000000,30.000000,30.000000,30.00,30.000000,19.000000,30.000000,30.000000,30.00,19.000000


# 요약문 실험 테스트 수행

## 1. original 본문 가져오기

In [297]:
test_ticker = '우리금융지주'

In [298]:
CrossEncoder_prompt_test = f'''
이 뉴스들 중에서 "{test_ticker}"가 핵심 주제로 다뤄진 기사만 알려줘.
다른 회사 언급이 많거나, {test_ticker}가 단순히 함께 언급된 기사라면 제외해줘.
'''

In [312]:
retriever_test = vectordb.as_retriever(search_kwargs={"k": 50,'filter' : {'ticker' : test_ticker}})
compressor = CrossEncoderReranker(model=model, top_n=50)

In [313]:
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever_test
)

In [315]:
compressed_docs = compression_retriever.invoke(CrossEncoder_prompt_test)

/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [316]:
def get_full_article_from_chroma(original_idx: int, kind : str, vectordb) -> dict:
    """original_idx 기준으로 chunk들을 모아 원문 복원"""
    # 1. 해당 article의 모든 chunk 가져오기
    VecDB = vectordb._collection
    results = VecDB.get(
      where = { "$and" : [ # 빈 쿼리로 전체 탐색
            {"original_idx": original_idx}, 
            {"ticker": kind}
            ]
         },
         include = ['documents','metadatas']   
        )
        
    if not results:
        return {"error": f"No chunks found for original_idx {original_idx}"}

    #print(results)
    #print(results['metadatas'][0]['chunk_idx'])

    # 4. 대표 metadata 하나 뽑아 저장
    return {
        "title": results['metadatas'][0]["title"],
        "url": results['metadatas'][0]["url"],
        "Date": results['metadatas'][0]["Date"],
        "ticker": results['metadatas'][0]["ticker"],
        "content": results['documents'][0]
    }

In [317]:
original_idxs = list(map(lambda x: x.metadata['original_idx'], compressed_docs))

In [318]:
top_n_original = [get_full_article_from_chroma(idx,kind=test_ticker,vectordb=vectordb) for idx in original_idxs]

In [320]:
top_n_original[:5]

[{'title': "'우리금융지주' 52주 신고가 경신, 전일 외국인 대량 순매수",
  'url': 'https://www.hankyung.com/article/202507145874L',
  'Date': '2025-07-14',
  'ticker': '우리금융지주',
  'content': '◆ 주체별 매매동향- 전일 외국인 대량 순매수지난 한달을 기준으로 보면 외국인이 296.2만주를 순매수했고, 개인들도 41.8만주를 순매수했다. 하지만 기관은 294.9만주를 순매도했다. 같은 기간 이 종목의 거래 비중은 외국인과 개인이 각각 51.3%, 26.2%로 비중이 높다.더욱이 전일 외국인이 대량 순매수를 하고 있어 투자자들의 관심이 집중되고 있다.[그래프]우리금융지주 외국인/기관 매매동향\n\n\n\n\n\n\n\n\n\n\n\n◆ 최근 애널리스트 분석의견- 갈수록 돋보일 고배당 매력 - NH투자증권, BUY07월 09일 NH투자증권의 정준섭 애널리스트는 우리금융지주에 대해 "\'25년 2분기 지배순이익은 다소 부진할 예정. 경상 실적(이자이 익, 비이자이익)은 견조하나, 책준형 신탁 충당금 등으로 대손비용률 상승 (65bp, +20bp q-q, +21bp y-y)이 예상되기 때문. 우리금융지주는 하반기 자사주 매입 기대감은 낮지만, 배당 매력은 갈수록 돋보일 전망"이라고 분석하며, 투자의견 \'BUY\', 목표주가 \'29,000원\'을 제시했다.한경로보뉴스이 기사는 한국경제신문과 금융 AI 전문기업 씽크풀이 공동 개발한 기사 자동생성 알고리즘에 의해 실시간으로 작성된 것입니다.'},
 {'title': "'우리금융지주' 52주 신고가 경신, 전일 외국인 대량 순매수",
  'url': 'https://www.hankyung.com/article/202507084010L',
  'Date': '2025-07-08',
  'ticker': '우리금융지주',
  'content': '◆ 주체별 매매동향- 전일 외국인 대량 순매수지난 한달을 기준으로 보면 외

In [321]:
original_contents = list(map(lambda x: x['content'],top_n_original))

In [322]:
total_contents = ''.join(original_contents)

In [323]:
total_contents

'◆ 주체별 매매동향- 전일 외국인 대량 순매수지난 한달을 기준으로 보면 외국인이 296.2만주를 순매수했고, 개인들도 41.8만주를 순매수했다. 하지만 기관은 294.9만주를 순매도했다. 같은 기간 이 종목의 거래 비중은 외국인과 개인이 각각 51.3%, 26.2%로 비중이 높다.더욱이 전일 외국인이 대량 순매수를 하고 있어 투자자들의 관심이 집중되고 있다.[그래프]우리금융지주 외국인/기관 매매동향\n\n\n\n\n\n\n\n\n\n\n\n◆ 최근 애널리스트 분석의견- 갈수록 돋보일 고배당 매력 - NH투자증권, BUY07월 09일 NH투자증권의 정준섭 애널리스트는 우리금융지주에 대해 "\'25년 2분기 지배순이익은 다소 부진할 예정. 경상 실적(이자이 익, 비이자이익)은 견조하나, 책준형 신탁 충당금 등으로 대손비용률 상승 (65bp, +20bp q-q, +21bp y-y)이 예상되기 때문. 우리금융지주는 하반기 자사주 매입 기대감은 낮지만, 배당 매력은 갈수록 돋보일 전망"이라고 분석하며, 투자의견 \'BUY\', 목표주가 \'29,000원\'을 제시했다.한경로보뉴스이 기사는 한국경제신문과 금융 AI 전문기업 씽크풀이 공동 개발한 기사 자동생성 알고리즘에 의해 실시간으로 작성된 것입니다.◆ 주체별 매매동향- 전일 외국인 대량 순매수지난 한달을 기준으로 보면 외국인이 490.4만주를 순매수했고, 개인들도 15.8만주를 순매수했다. 하지만 기관은 470.8만주를 순매도했다. 같은 기간 이 종목의 거래 비중은 외국인과 개인이 각각 52.6%, 24.3%로 비중이 높다.한편 외국인은 이 종목에 대해서 최근 3일 연속 29.4만주 순매수를 하고 있다. 더욱이 전일 외국인이 대량 순매수를 하고 있어 투자자들의 관심이 집중되고 있다.[그래프]우리금융지주 외국인/기관 매매동향\n\n\n\n\n\n\n\n\n\n\n\n◆ 최근 애널리스트 분석의견- 2Q25 Preview: 시간이 더 필요하다 - 상상인증권, BUY(신규)07월 03일 상상인증권의 김현수 애널리스트는 우리

In [324]:
len(total_contents)

22496

수기 입력으로 확인
> 확인 사유 : 0번째 문서가 이상하..?

In [310]:
retriever = vectordb.as_retriever(search_kwargs={"k": 50,'filter' : {'ticker' : test_ticker}})
CrossEncoder_prompt = f'''
이 뉴스들 중에서 "{test_ticker}"가 핵심 주제로 다뤄진 기사만 알려줘.
다른 회사 언급이 많거나, {test_ticker}가 단순히 함께 언급된 기사라면 제외해줘.
'''
print(CrossEncoder_prompt.strip())
raw_docs = retriever.get_relevant_documents(CrossEncoder_prompt)
pairs = [(CrossEncoder_prompt, d.page_content) for d in raw_docs]

scores = model.score(pairs)
scored = sorted(zip(raw_docs, scores), key=lambda x: float(x[1]), reverse=True)[:50]
scored_docs=[]
# 3) 점수 붙이고 재정렬
for d, s in zip(raw_docs, scores):
    dd = deepcopy(d)
    dd.metadata["relevance_score"] = float(s)
    scored_docs.append(dd)

이 뉴스들 중에서 "우리금융지주"가 핵심 주제로 다뤄진 기사만 알려줘.
다른 회사 언급이 많거나, 우리금융지주가 단순히 함께 언급된 기사라면 제외해줘.


/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [311]:
pd.DataFrame(list(map(lambda x: x.metadata,scored_docs))).sort_values(by=['ticker','relevance_score'],ascending=False).head(5)

,chunk_idx,title,url,label,original_idx,ticker,Date,relevance_score
3,0,"'우리금융지주' 52주 신고가 경신, 전일 외국인 대량 순매수",https://www.hankyung.com/article/202507145874L,1,237,우리금융지주,2025-07-14,0.602175
14,0,"'우리금융지주' 52주 신고가 경신, 전일 외국인 대량 순매수",https://www.hankyung.com/article/202507084010L,1,374,우리금융지주,2025-07-08,0.467356
9,0,"""우리금융지주, 연말로 갈수록 고배당 부각…목표가↑""-NH",https://www.hankyung.com/article/2025070836976,1,476,우리금융지주,2025-07-08,0.397002
6,1,대출 수익성 악화에…4대 금융 실적 꺾였다,https://www.hankyung.com/article/2025071501731,1,158,우리금융지주,2025-07-15,0.358331
10,0,"“금리 하락에도 이익 증가” 금융지주, 실적 온도차",https://magazine.hankyung.com/business/article...,1,446,우리금융지주,2025-07-21,0.265313


## 요약하기

### 요약함수 호출
- 가져온 원 본문을 전부 적용하기

In [119]:
from summart_function_openai_2 import NewsSummaryAgent, build_summary_graph  # 너가 만든 것

In [124]:
def summarize_top_articles_2(total_contents: str,ticker:str,max_iters:int) -> pd.DataFrame:
    agent = NewsSummaryAgent(max_iters=max_iters)
    runnable = build_summary_graph(agent)

    rows = []
    doc = total_contents

    state = {"article": doc, "summary": "", "feedback": "", "iteration": 0}
    result = runnable.invoke(state)

    print("✅ 실행 결과 키:", result.keys())
    # 여기서 final_summary 반드시 존재해야 함(위 패치 기준)
    final_summary = result.get("final_summary")
    if not final_summary:
        print("❌ final_summary 없음. 디버그용 전체 상태:", result)
        # 계속 진행할지, 실패로 표기할지 선택

    rows.append({
        "ticker": ticker,
        "date": '2025-07-30', # 위 조회 기준일자로 연동시켜서 바꿀 예정
        "summary": final_summary,
        "feedback": result.get("last_feedback", "피드백 없음"),
    })

    return pd.DataFrame(rows)


In [325]:
result_df = summarize_top_articles_2(total_contents,ticker=test_ticker,max_iters=5)

summart : ✅ 주요 요약
- 외국인, 우리금융지주 대량 순매수
  외국인은 최근 3일 연속으로 우리금융지주 주식을 대량 순매수하며 투자자들의 관심이 집중되고 있다.

- 우리금융지주, 2분기 순이익 감소 전망
  우리금융지주의 2분기 순이익은 전년 대비 감소할 것으로 예상되며, 이는 모바일 트레이딩 시스템 개발과 신규 인력 채용 등으로 인한 판매관리비 증가가 원인으로 분석된다.

- 4대 금융지주, 비이자이익 증가로 실적 선방
  KB, 신한, 하나, 우리 등 4대 금융지주는 비이자이익 증가 덕분에 2분기 실적이 개선되었으나, 하반기에는 경기 침체와 대출 자산 확대 어려움으로 실적이 나빠질 우려가 있다.

🔑 키워드: 외국인 순매수, 우리금융지주, 2분기 순이익, 비이자이익, 4대 금융지주, 경기 침체, 대출 자산, 실적 전망.
[should_stop] Iteration: 0
[should_stop] Feedback:
 - 정확성: 부족함 <reason> [요약에서 언급된 외국인 순매수와 관련된 정보는 기사에 있지만, 요약에서 언급된 '우리금융지주 2분기 순이익 감소 전망'의 구체적인 원인인 '모바일 트레이딩 시스템 개발과 신규 인력 채용'은 기사에 명시되어 있지 않습니다.] </reason>
- 포괄성: 부족함 <reason> [요약은 외국인 순매수와 2분기 실적 전망에 대한 주요 내용을 다루고 있지만, 기사에 포함된 다양한 애널리스트 의견과 금융지주들의 배당 및 자사주 매입 관련 정보가 누락되었습니다.] </reason>
- 간결성: 좋음 <reason> [요약은 불필요한 표현 없이 핵심 정보를 간결하게 전달하고 있습니다.] </reason>
- 문장구성: 좋음 <reason> [문장이 자연스럽고 명확하게 구성되어 있어 이해하기 쉽습니다.] </reason>
- 일관성: 부족함 <reason> [요약은 특정 종목인 '우리금융지주'에 대한 내용에 집중하고 있지만, 기사에는 다른 금융지주들에 대한 정보도 포함되어 있어 일관성이 부족합니다.] </reas

In [326]:
result_df

,ticker,date,summary,feedback
0,우리금융지주,2025-07-30,"✅ 주요 요약\n\n- **외국인, 우리금융지주 대량 순매수**\n - 외국인은 ...",- 정확성: 부족함 <reason> [요약에서 언급된 일부 정보가 원문 기사와 일치...


In [327]:
print(result_df.summary.values[0])

✅ 주요 요약

- **외국인, 우리금융지주 대량 순매수**
  - 외국인은 최근 3일 동안 우리금융지주 주식을 29.4만주 순매수하며 투자자들의 주목을 받고 있습니다. 지난 한 달 동안 외국인은 490.4만주를 순매수한 반면, 기관은 470.8만주를 순매도했습니다.

- **우리금융지주, 2분기 순이익 감소 전망**
  - 우리금융지주의 2분기 순이익은 전년 대비 8.6% 감소한 8784억원으로 예상됩니다. 이는 모바일 트레이딩 시스템 개발과 신규 인력 채용에 따른 판매관리비 증가가 주요 원인으로 분석됩니다.

- **4대 금융지주, 비이자이익 증가로 실적 선방**
  - KB, 신한, 하나, 우리 등 4대 금융지주는 비이자이익 증가 덕분에 2분기 실적이 개선되었습니다. 그러나 하반기에는 경기 침체와 대출 자산 확대 어려움으로 실적이 나빠질 우려가 있습니다.

- **애널리스트 의견 및 배당 매력**
  - NH투자증권과 상상인증권은 우리금융지주의 배당 매력을 강조하며 투자의견을 '매수'로 유지했습니다. 하반기 자사주 매입 가능성은 낮지만, 배당 매력은 더욱 부각될 전망입니다.

- **금융지주들의 배당 및 자사주 매입**
  - 4대 금융지주는 하반기 자사주 매입 및 소각 규모가 최소 1조 6천억 원에 이를 것으로 예상되며, 주주친화 정책 강화가 주가 상승의 모멘텀으로 작용하고 있습니다.

- **다른 금융지주사들의 실적 및 전망**
  - 신한금융지주는 2분기 순이익이 1.3% 증가할 것으로 예상되며, 하나금융지주는 7% 이상 증가할 것으로 보입니다. 이는 신용카드, 증권 중개, 운용리스 등 수수료 수익이 양호한 흐름을 이어가고 있기 때문입니다.

- **금융지주사들의 장기 전망**
  - 4대 금융지주의 연간 순이익은 총 18조원에 육박할 것으로 예상되며, 이는 비이자이익 증가와 주주환원 정책 강화에 기인합니다. 그러나 경기 침체와 대출 규제 강화로 인해 하반기 실적에는 불확실성이 존재합니다.

🔑 **키워드**: 외국인 순매수, 우리금융지주, 2분기 

## 피드백 정리
- 삼성전자  : 삼성전자에 대한 내용 잘 요약
- 우리금융지주 : 우리금융지주에 대한 내용 잘 요약
- SK하이닉스 : 요약은 되었으나,, 시장 내/외부 사람들의 구매 패턴, 동향에 대한 정보가 주로 요약이 되고 있음<br>
            > recall@50(pool) 이 상당히 낮게 나옴!
  > SK하이닉스에 대한 케이스를 봤을 때, chunk에 대해서만 요약을 해야하나 싶음<br>
  > 2025.08.11) 현재 작업은 전체 원문에 대한 요약이었음. 그래서.. SK 하이닉스 회사 본질에 대한 정보를 제대로 요약해주지 못하나 싶음<br>
  > 프롬프트 내부 요약 평가 기준 의 "일관성" 영역에 명확히 종목명을 말해야겠다..

## 2. chunk만 수행하기

In [ ]:
test_ticker = '우리금융지주'
CrossEncoder_prompt_test = f'''
이 뉴스들 중에서 "{test_ticker}"가 핵심 주제로 다뤄진 기사만 알려줘.
다른 회사 언급이 많거나, {test_ticker}가 단순히 함께 언급된 기사라면 제외해줘.
'''

In [254]:
#model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base")
model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")
compressor = CrossEncoderReranker(model=model, top_n=50)

In [255]:
retriever_test = vectordb.as_retriever(search_kwargs={"k": 50,'filter' : {'ticker' : test_ticker}})

In [256]:
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever_test
)

In [257]:
compressed_docs = compression_retriever.invoke(CrossEncoder_prompt)

/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [259]:
list(map(lambda x: x.metadata['title'],compressed_docs))[:5]

["SK하이닉스 또 '사상 최고' 실적…분기 영업익 9조 넘었다",
 "SK하이닉스 사상 최대 실적에도…'신중론' 여전한 이유 [종목+]",
 "'역대급 실적' SK하이닉스, 2분기 영업이익 9조원 넘었다",
 'AI 수요 폭증과 HBM 기술 주도에 힘입은 SK하이닉스, 주가 반등 흐름 가속화',
 'SK하이닉스, 삼성전자 제치고 ‘대학생이 일하고 싶은 기업’ 첫 1위']

compressed_docs와 수기로 계산한 것이 같은 것 증명

In [246]:
retriever = vectordb.as_retriever(search_kwargs={"k": 50,'filter' : {'ticker' : test_ticker}})
CrossEncoder_prompt = f'''
이 뉴스들 중에서 "{test_ticker}"가 핵심 주제로 다뤄진 기사만 알려줘.
다른 회사 언급이 많거나, {test_ticker}가 단순히 함께 언급된 기사라면 제외해줘.
'''
print(CrossEncoder_prompt.strip())
raw_docs = retriever.get_relevant_documents(CrossEncoder_prompt)
pairs = [(CrossEncoder_prompt, d.page_content) for d in raw_docs]

scores = model.score(pairs)
scored = sorted(zip(raw_docs, scores), key=lambda x: float(x[1]), reverse=True)[:50]
scored_docs=[]
# 3) 점수 붙이고 재정렬
for d, s in zip(raw_docs, scores):
    dd = deepcopy(d)
    dd.metadata["relevance_score"] = float(s)
    scored_docs.append(dd)

이 뉴스들 중에서 "SK하이닉스"가 핵심 주제로 다뤄진 기사만 알려줘.
다른 회사 언급이 많거나, SK하이닉스가 단순히 함께 언급된 기사라면 제외해줘.


/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `XLMRobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [260]:
pd.DataFrame(list(map(lambda x: x.metadata,scored_docs))).sort_values(by=['ticker','relevance_score'],ascending=False).head(5)

,original_idx,title,url,label,chunk_idx,ticker,Date,relevance_score
47,3560,SK하이닉스 또 '사상 최고' 실적…분기 영업익 9조 넘었다,https://www.hankyung.com/article/202507248226g,1,1,SK하이닉스,2025-07-24,0.684618
25,3927,SK하이닉스 사상 최대 실적에도…'신중론' 여전한 이유 [종목+],https://www.hankyung.com/article/2025072510306,1,1,SK하이닉스,2025-07-25,0.672207
37,3724,"'역대급 실적' SK하이닉스, 2분기 영업이익 9조원 넘었다",https://magazine.hankyung.com/business/article...,1,0,SK하이닉스,2025-07-24,0.611750
15,3744,"AI 수요 폭증과 HBM 기술 주도에 힘입은 SK하이닉스, 주가 반등 흐름 가속화",https://www.hankyung.com/article/202507313103a,1,0,SK하이닉스,2025-07-31,0.601519
14,3867,"SK하이닉스, 삼성전자 제치고 ‘대학생이 일하고 싶은 기업’ 첫 1위",https://magazine.hankyung.com/job-joy/article/...,1,2,SK하이닉스,2025-07-28,0.455363


이어서 시작

In [336]:
unique_lst = list(set(list(map(lambda x: x.page_content,compressed_docs))))

In [337]:
chunk_total = ''.join(unique_lst)

In [338]:
chunk_total

'KB 신한 하나 우리 등 4대 금융지주의 올해 2분기 합산 순이익이 전년 동기 대비 감소한 것으로 파악됐다. 금융지주의 실적이 전년 동기 대비 감소한 것은 홍콩 H지수 주가연계증권(ELS) 배상으로 1조원 넘는 일회성 비용이 발생한 2024년 1분기를 제외하면 1년 반 만에 처음이다. 금융회사의 핵심 수익원인 이자수익이 줄줄이 감소한 결과다. 금융지주의 핵심 자회사인 은행들이 가계대출 억제 정책과 경기 침체로 대출 자산을 확대하기 어려운 만큼 향후 금융지주의 실적 감소세가 본격화할 것이란 관측이 제기된다.\n                    \n\n\n\n\n\n\n이미지 크게보기증가로 순이익이 지난해 3조 1715억원에서 올해 3조 1095억원으로 소폭 줄어들 것으로 예상된다.4대 금융지주의 순이익 합계는 지난해 16조 5268억원에서 올해 17조 8250억원으로 8% 가까이 증가할 전망이다.정부의 고강도 부동산 대출 규제에도 불구하고 주요 금융지주들의 연간 실적 전망은 오히려 상향 조정되는 추세다.이는 각 금융지주가 이자이익 축소에 대비해 새로운 수익원 확보에 주력했기 때문으로 분석된다.KB금융은 오는 24일, 신한·하나·우리금융은 25일 차례로 2분기 실적을 발표할 예정이다.정유진 기자 jinjin@hankyung.com강세장을 이끈 것은 5조원 순매수한 연기금 외에 자사주와 외국인이다. 현재까지 자사주 매입은 약 10조원에 달할 것으로 추정되는데 삼성전자와 은행이 주요 매수 주체였다. 이 추세는 정부 정책과 맞물려서 하반기에도 이어진다. 하반기 6조원이 더해지면 올해 약16조원이 자사주 매입의 형태로 증시에 유입되는데 이는 작년의 2조원 자사주 순매수와 크게 대비된다. 외인은 연초부터 4월 말까지 트럼프의 관세 악재로 18조 매도했지만 이후 매수로 전환해서 12조원의 순매수를 누적했다.2000년 이후 25년 동안 외인의 누적 순매수 추세는 2번의 80조원 싸이클과 1번의 40조원 사이클을 보인 후 4번째 순환사이클을 막 시작했다. 직전 순매수 저점에

In [339]:
len(chunk_total)

37824

In [340]:
result_df_chunk = summarize_top_articles_2(chunk_total,ticker=test_ticker,max_iters=5)

summart : ✅ 주요 요약
- 4대 금융지주, 2분기 순이익 전년 대비 감소
  KB, 신한, 하나, 우리 등 4대 금융지주의 2분기 합산 순이익이 전년 동기 대비 감소했으며, 이는 이자수익 감소와 가계대출 억제 정책으로 인한 결과로 분석됨.

- 금융지주, 주주환원 정책 강화
  4대 금융지주는 자사주 매입 및 소각을 포함한 주주환원 정책을 강화하고 있으며, 이는 주가 상승과 주주가치 제고에 긍정적인 영향을 미칠 것으로 예상됨.

- 정부의 부동산 대출 규제와 증시 친화 정책
  정부의 부동산 대출 규제에도 불구하고 금융지주들의 실적 전망은 상향 조정되고 있으며, 이는 새로운 수익원 확보와 증시 친화적인 정책 덕분으로 분석됨.

🔑 키워드
- 4대 금융지주
- 순이익 감소
- 이자수익
- 가계대출 억제
- 주주환원 정책
- 자사주 매입
- 부동산 대출 규제
- 증시 친화 정책
[should_stop] Iteration: 0
[should_stop] Feedback:
 - 정확성: 부족함 <reason> [요약 내용이 원문 기사와 일부 일치하지 않으며, 특히 금융지주의 순이익 증가에 대한 부분이 잘못 전달됨] </reason>
- 포괄성: 부족함 <reason> [기사의 중요한 정보 중 일부가 누락되었으며, 특히 정부의 정책 변화와 이에 따른 금융지주의 대응 전략이 충분히 다루어지지 않음] </reason>
- 간결성: 좋음 <reason> [요약은 간결하게 작성되어 있으며, 불필요한 정보가 포함되지 않음] </reason>
- 문장구성: 좋음 <reason> [문장이 자연스럽고 명확하게 구성되어 있음] </reason>
- 일관성: 부족함 <reason> [특정 종목에 대한 내용이 아닌, 전체 금융지주에 대한 내용임에도 불구하고 일관성이 부족함] </reason>

피드백:
1. 정확성을 높이기 위해 원문 기사에서 언급된 순이익 증가와 감소에 대한 정보를 정확히 반영해야 합니다. 특히, 금융지주의 순이익이 전년 대비 증가했다는 부분을 명확히 해야 합니다

KeyboardInterrupt: 

In [270]:
print(result_df_chunk.summary.values[0])

✅ 주요 요약

- **SK하이닉스, 2분기 사상 최대 실적 기록**
  SK하이닉스는 2분기 매출 22조2320억원, 영업이익 9조2129억원을 기록하며 사상 최대 실적을 달성했다. 이는 AI 메모리 수요 증가와 D램 및 낸드플래시 출하량 증가에 기인한다.

- **HBM 시장에서의 경쟁 심화 우려**
  SK하이닉스는 HBM 시장에서의 독점적 지위가 경쟁 심화로 인해 위협받을 수 있다는 우려가 제기되고 있다. 삼성전자와 마이크론의 시장 진입이 예상되며, 가격 하락 가능성이 논의되고 있다.

- **SK하이닉스의 HBM 수요 성장 확신**
  SK하이닉스는 HBM 수요가 지속적으로 증가할 것이라고 확신하며, AI 시장에서의 핵심 제품으로서의 중요성을 강조하고 있다. 이에 따라 HBM 생산을 위한 투자를 확대할 계획이다.

- **주가 변동 및 외부 기관의 평가**
  SK하이닉스의 실적 발표 이후 주가는 일시적으로 상승했으나, 외부 기관의 평가에 따라 변동성을 보였다. 골드만삭스는 HBM 시장의 경쟁 심화로 인한 가격 하락 가능성을 지적하며 투자의견을 하향 조정했다. 반면, 다른 기관들은 SK하이닉스의 기술력과 시장 지배력을 긍정적으로 평가하며, 장기적인 성장 가능성을 강조했다.

🔑 키워드
- SK하이닉스
- 2분기 실적
- HBM 시장
- AI 메모리
- 경쟁 심화
- 가격 하락
- 투자 확대
- 주가 변동
- 외부 기관 평가


## 원문 그대로 vs chunking만 사용 비교
> 결론 : chunking된 문서만을 사용하는 게 유리?

[SK하이닉스]

1. 원문 그대로
- **외국인 투자자, 전기·전자 및 운수장비 업종에 집중 매수 및 매도**  <br>
  외국인 투자자들은 한화오션, SK하이닉스, 두산에너빌리티 등을 중점적으로 매수했으며, SK하이닉스, 카카오페이, NAVER 등을 매도했습니다. 특히, 전기·전자 업종에 속한 종목들이 다수 포함되어 있으며, 이들 종목의 주가 변동이 투자 전략에 영향을 미쳤습니다.<br>

- **기관 투자자, 전기·전자 및 운수장비 업종에 집중**  <br>
  기관 투자자들은 삼성전자, 현대차, 삼성전기 등 전기·전자 업종의 종목을 중점적으로 매수했으며, SK하이닉스, KODEX 200선물인버스2X 등을 매도했습니다. <br>이들 종목 중 일부는 전일 대비 주가가 상승했습니다.

- **코스피지수 상승, 투자자 고민 증가**  <br>
  코스피지수가 3200을 넘어섰지만, 주식 매입 시점을 고민하는 투자자들이 증가하고 있습니다. 기존 대장주들의 추가 상승 가능성과 이미 충분히 올랐다는 부담감이 동시에 작용하고 있습니다.

- **주요 종목의 주가 변동**  <br>
  한화오션, SK하이닉스, 두산에너빌리티 등은 전일 대비 주가가 상승했으며, 기아, 삼성전자, 현대차 등은 주가가 하락했습니다. 이러한 변동은 투자자들의 매수 및 매도 전략에 영향을 미치고 있습니다.


2. chunking된 문서만 사용
- **SK하이닉스, 2분기 사상 최대 실적 기록**<br>
  SK하이닉스는 2분기 매출 22조2320억원, 영업이익 9조2129억원을 기록하며 사상 최대 실적을 달성했다. 이는 AI 메모리 수요 증가와 D램 및 낸드플래시 출하량 증가에 기인한다.<br>

- **HBM 시장에서의 경쟁 심화 우려**<br>
  SK하이닉스는 HBM 시장에서의 독점적 지위가 경쟁 심화로 인해 위협받을 수 있다는 우려가 제기되고 있다. 삼성전자와 마이크론의 시장 진입이 예상되며, 가격 하락 가능성이 논의되고 있다.<br>

- **SK하이닉스의 HBM 수요 성장 확신**<br>
  SK하이닉스는 HBM 수요가 지속적으로 증가할 것이라고 확신하며, AI 시장에서의 핵심 제품으로서의 중요성을 강조하고 있다. 이에 따라 HBM 생산을 위한 투자를 확대할 계획이다.

- **주가 변동 및 외부 기관의 평가**<br>
  SK하이닉스의 실적 발표 이후 주가는 일시적으로 상승했으나, 외부 기관의 평가에 따라 변동성을 보였다. 골드만삭스는 HBM 시장의 경쟁 심화로 인한 가격 하락 가능성을 지적하며 투자의견을 하향 조정했다. 반면, 다른 기관들은 SK하이닉스의 기술력과 시장 지배력을 긍정적으로 평가하며, 장기적인 성장 가능성을 강조했다.<br>



# 과거 버전

In [204]:
import pandas as pd
from datetime import datetime, timedelta
# cross-encoder
from sentence_transformers import CrossEncoder
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

from summary_function_openai import NewsSummaryAgent, build_summary_graph  # 너가 만든 것

# 1. 모델 준비 (CrossEncoder for Re-ranking) # 예시 모델 하나 생성
rerank_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# 2. 최근 5일치 필터링 함수
def get_recent_articles(df: pd.DataFrame, ticker: str, days: int = 5):
    df['Date'] = pd.to_datetime(df['Date'])
    today = df['Date'].max()
    recent_df = df[
        (df['ticker'] == ticker) &
        (df['Date'] >= today - timedelta(days=days))
    ]
    return recent_df.sort_values(by="Date", ascending=False)

# 3. huggingface 모델 이용.
def rerank_articles(df: pd.DataFrame,ticker:str ,query: str, top_k: int = 5):
    task_df= df[df.ticker == ticker]
    docs = task_df['content'].tolist()
    # 모델 이용?? 
    pairs = [(query, doc) for doc in docs]
    scores = rerank_model.predict(pairs)
    
    task_df_2 = task_df.copy()
    task_df_2['score'] = scores
    return task_df_2.sort_values(by='score', ascending=False).head(top_k)

# 4. 전체 요약 실행 함수
def summarize_top_articles(df: pd.DataFrame, ticker: str, query: str, top_k: int = 5):
    recent_df = get_recent_articles(df, ticker)
    top_df = rerank_articles(recent_df, query=query,ticker=ticker, top_k=top_k)

    agent = NewsSummaryAgent()
    runnable = build_summary_graph(agent)

    results = []
    for _, row in top_df.iterrows():
        state = {
            "article": row['content'],
            "summary": "",
            "feedback": "",
            "iteration": 0
        }
        result = runnable.invoke(state)


        print("✅ 실행 결과 타입:", type(result))
        print("✅ 실행 결과 키 목록:", result.keys())
        print("✅ 실행 결과 전체 내용:", result)

        if "final_summary" not in result:
            print("❌ final_summary 키가 없습니다. 중단합니다.")
            continue  # 또는 raise Exception("final_summary 없음")

        print(f'실행 결과 : {result}')
        results.append({
            "ticker": row['ticker'],
            "date": row['Date'],
            "header": row['header'],
            "url": row['url'],
            "summary": result["final_summary"], 
             "feedback": result.get("last_feedback", "피드백 없음")
        })
    return pd.DataFrame(results)


In [55]:
query = "우리금융지주 관련 시황"
ticker = "우리금융지주"  # 예시
result_df = summarize_top_articles(cusA_news_df, ticker=ticker, query=query, top_k=3)

/Users/Work/SU/Project/streamlit/Streamlit/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


[should_stop] Iteration: 1
[should_stop] Feedback:
 - 정확성: 좋음 <reason> 원문의 내용을 정확하게 반영하고 있음. </reason>
- 포괄성: 부족함 <reason> 원문에서 언급된 '기술 경쟁력 확보, 고객사 확대, 수익구조 개선 등 실질적 사업성과로 뒷받침돼야 한다는 지적' 등의 중요한 내용이 누락되었음. </reason>
- 간결성: 좋음 <reason> 불필요한 표현 없이 요약 내용을 간결하게 전달하였음. </reason>
- 문장구성: 좋음 <reason> 문장이 자연스럽고 명확하게 구성되어 있음. </reason>

[피드백]
요약의 포괄성이 부족한 점이 아쉽습니다. 원문에서 언급된 '기술 경쟁력 확보, 고객사 확대, 수익구조 개선 등 실질적 사업성과로 뒷받침돼야 한다는 지적' 등의 중요한 내용을 요약에 포함시키면 더욱 완벽한 요약이 될 것 같습니다. 이 부분을 고려하여 요약을 수정해보시는 것을 추천드립니다.
✅ '정확성' 평가 통과
❌ '포괄성' 평가에서 좋음이 아님
[should_stop] next_step = no


KeyError: 'Input to PromptTemplate is missing variables {\'"foo"\', \'"properties"\'}.  Expected: [\'"foo"\', \'"properties"\', \'article\', \'feedback\', \'summary\'] Received: [\'article\', \'summary\', \'feedback\']\nNote: if you intended {"foo"} to be part of the string and not a variable, please escape it with double curly braces like: \'{{"foo"}}\'.\nFor troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/INVALID_PROMPT_INPUT '

In [35]:
def summarize_top_articles(total_contents :str):

    agent = NewsSummaryAgent()
    runnable = build_summary_graph(agent)
    results = []
    
    doc = total_contents

    state = {
        "article": doc['content'],
        "summary": "",
        "feedback": "",
        "iteration": 0
    }
    result = runnable.invoke(state)


    print("✅ 실행 결과 타입:", type(result))
    print("✅ 실행 결과 키 목록:", result.keys())
    print("✅ 실행 결과 전체 내용:", result)

    if "final_summary" not in result:
        print("❌ final_summary 키가 없습니다.")

    print(f'실행 결과 : {result}')
    results.append({
        "ticker": doc['ticker'],
        "date": doc['Date'],
        "header": doc['title'],
        "url": doc['url'],
        "summary": result["final_summary"], 
            "feedback": result.get("last_feedback", "피드백 없음")
    })
    return pd.DataFrame(results)
